In [ ]:
!pip install umap-learn

# Ensemble Clustering Pipeline
**T2D Dataset — KMeans · GMM · Agglomerative · Spectral · Fuzzy C-Means · TwoStep**

**Purpose.** Derive reproducible metabolic phenotypes of type 2 diabetes by combining six
clustering algorithms into a single weighted consensus, select the number of phenotypes K
from the ensemble itself, test how sensitive that choice is, and validate the resulting
partition statistically. The consensus labels exported here are the input to the survival
analysis.

**Input.** `t2d_fast_clean.csv` — the analytical cohort from the EDA notebook. 

**Outputs.** `consensus_k{K}_clusters.csv` (cohort plus consensus and per-algorithm labels),
figures in `figures/`, and the k-sensitivity supplement in `k_sensitivity_outputs/`.

**How to run.** Top to bottom, from the folder containing the input CSV. Cells are not
independently runnable: every stage consumes objects built by the previous one. Requires
`numpy`, `pandas`, `scikit-learn`, `scipy`, `scikit-fuzzy`, `hdbscan`, `joblib`, `seaborn`,
`matplotlib` and optionally `umap-learn` (the notebook degrades gracefully without it).
Reproducibility rests on the single constant `SEED = 42`.

---

## Feature space

Five features — `Age`, `BMI`, `HbA1c`, `HOMA_IR`, `HOMA_B` — matching the Ahlqvist-style
variable set (age, adiposity, glycaemia, insulin resistance, beta-cell function). HOMA-IR and
HOMA-β are log-transformed because they are strictly positive with long right tails; all five
are then standardised to mean 0 / SD 1 so that no feature dominates the Euclidean distances
purely through its unit scale.

---

## Pipeline logic

The pipeline exists to answer a problem single-algorithm clustering cannot: **different
algorithms give different answers, and there is no ground truth to arbitrate between them.**
The response here is to treat inter-algorithm agreement as the evidence.

| Stage | What happens | Why |
|---|---|---|
| 1 | Six algorithms each choose their own K by their own internal criterion (silhouette, BIC, Xie-Beni, eigengap) | Their disagreement is the raw material, not a defect |
| 2 | Each algorithm is weighted by (mean peer ARI + silhouette), softmax-normalised, then combined into a weighted co-association matrix `C` | An algorithm earns influence by being both internally coherent and reproducible by its peers |
| 3 | `C` is converted to a distance (`1 - C`), one average-linkage hierarchy `Z_cons` is built, and each candidate K is scored on four axes: silhouette, weighted ARI, intra-cluster co-association, spectral eigengap | Four criteria because each is blind in a different way; one shared tree so all K values are comparable cuts of the same structure |
| 4 | All six algorithms are re-run at the consensus K | Stage-1 partitions used different K and cannot be compared like-for-like |
| 4b | `Z_cons` is re-cut at k = 2, 3, 4, 5 | Tests whether the published k=3 partition is empirically supported, and specifically whether SOIRD splits into MOD-like and SIRD-like subtypes at k=4 |
| 5 | Each algorithm is ranked on silhouette, ARI vs consensus, peer Jaccard consistency, and bootstrap stability (B=30, 80% subsamples) | Robustness to resampling separates real structure from sample-specific artefacts |

`C[i, j]` is the weighted fraction of algorithms that placed samples *i* and *j* together — an
empirical similarity built from agreement rather than from feature-space geometry. It is the
central object of the whole method.

**On label bases.** `fcluster` returns 1-based labels, and Stage 4 shifts every sklearn output
from 0-based to 1-based to match, so that `CLUSTER_PALETTE` colours apply uniformly. This is
cosmetic: it changes label *values*, never the partition. All ARI-based comparisons are
unaffected because ARI is invariant to label permutation — which is also why the bootstrap can
compare re-clustered subsets against the originals without re-mapping.

---

## Known issues to review before use

Spotted while documenting and **left unchanged**, since the analysis logic was preserved as-is:

1. **Hard-coded phenotype mapping.** The comment block in Stage 4b Cell 2 fixes cluster 1 → SIDD
   (n≈527), 2 → MARD (n≈1850), 3 → SOIRD (n≈1544). `fcluster` numbering is not guaranteed
   stable if `C` changes, so re-verify this mapping against the printed Z-score profiles after
   any re-run before quoting it.
2. **`Q_stab` includes the diagonal.** In `find_best_k_from_consensus`, `same_mask` includes
   `i == j` pairs where `C[i, i] = 1.0` by construction. As K grows there are fewer same-cluster
   pairs, so the fixed N diagonal entries form a larger share of the mean and pull `Q_stab`
   upward. The bias favours *larger* K, so it works against the selected k=3 and is conservative
   here — but it is worth stating in the methods.
   



> Cell outputs have been cleared so the notebook is light and diffs cleanly. Re-run top to
> bottom to regenerate them.

---

## Notebook contents

| Section | Description |
|---------|-------------|
| 1 | Imports |
| 2 | Clustering algorithm definitions |
| 3 | Ensemble & ranking function definitions |
| 4 | Load & preprocess data |
| 5 | Stage 1 — Individual algorithm optimisation |
| 6 | Stage 2 — Weighted consensus matrix |
| 7 | Stage 3 — Optimal K selection |
| 8 | Stage 4 — Fixed-K re-run at consensus K |
| 8b | Stage 4b — k-sensitivity analysis |
| 9 | Stage 5 — Algorithm ranking |
| 10 | Visualisation functions |
| 11 | Cluster composition analysis & export |
| 12 | Bootstrap confidence intervals |
| 13 | Statistical validation — Kruskal-Wallis & post-hoc |


## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import warnings
# CAUTION: this hides ALL warnings, including sklearn convergence warnings
# (e.g. GMM failing to converge at a given K) and numeric RuntimeWarnings.
# Comment out when debugging or upgrading library versions.
warnings.filterwarnings('ignore')

import hdbscan                              # density-based clustering (defined but not used in Stage 1 — see notebook header)
import skfuzzy as fuzz                      # reference Fuzzy C-Means implementation used at fixed K

from joblib import Parallel, delayed          # NEW: parallel bootstrap

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    # ARI is permutation-invariant: it compares partitions, not label values.
    # That property is what lets us compare 0-based sklearn output against
    # 1-based consensus labels without any re-mapping.
    silhouette_score, adjusted_rand_score,
    calinski_harabasz_score, davies_bouldin_score  # NEW: extra internal indices
)
from sklearn.mixture import GaussianMixture
from sklearn.cluster import (
    KMeans, AgglomerativeClustering, SpectralClustering, Birch
)
from sklearn.neighbors import kneighbors_graph

from scipy.sparse import csgraph
from scipy.sparse.linalg import eigsh        # partial eigendecomposition (only the smallest few eigenvalues)
from scipy.spatial.distance import cdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment   # Hungarian matching for label alignment
from scipy.linalg import eigh                # dense fallback eigensolver
from scipy.stats import kruskal, mannwhitneyu  # NEW: statistical tests

# UMAP — graceful fallback if not installed
# Kept optional so the notebook runs end-to-end without umap-learn; the UMAP
# figures are simply skipped (or fall back to t-SNE) rather than raising.
try:
    import umap as umap_lib
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("  umap-learn not found — UMAP plots will be skipped.")
    print("  Install with:  pip install umap-learn")

print("All imports successful.")

## 2 · Clustering Algorithm Definitions

### 2a · Helper

In [ ]:
def report_clusters(df, labels, features, name):
    """Print a one-line cluster summary; return cluster count."""
    unique = np.unique(labels)
    # Exclude -1, which HDBSCAN uses for noise points that belong to no cluster.
    # Counting it would inflate K by one.
    n_clusters = len(unique[unique != -1])
    print(f"  [{name}]  K = {n_clusters}")
    return n_clusters

### 2b · Fuzzy C-Means  (pure numpy, auto-K via Xie-Beni)

In [ ]:
def _fuzzy_c_means_fixed(X, c, m=2.0, max_iter=200, tol=1e-5, random_state=42):
    """Deterministic Fuzzy C-Means with a local RandomState."""
    # A LOCAL RandomState (not np.random.seed) keeps this function reproducible
    # without mutating global random state that other algorithms depend on.
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    # U is the membership matrix (n samples x c clusters). Random init, then
    # each row is normalised so a sample's memberships sum to 1.
    U = rng.rand(n, c)
    U /= U.sum(axis=1, keepdims=True)
    centers = None
    for _ in range(max_iter):
        U_old = U.copy()
        um = U ** m            # m is the fuzzifier: higher m = softer boundaries
        # Cluster centre = membership-weighted mean of all points.
        centers = (um.T @ X) / um.sum(axis=0)[:, None]
        # Distance from every point to every centre. The +1e-10 prevents a
        # divide-by-zero when a point sits exactly on a centre.
        dist = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2) + 1e-10
        # Standard FCM membership update: membership falls off with relative
        # distance to each centre, controlled by the exponent 2/(m-1).
        U = 1.0 / np.sum(
            (dist[:, :, None] / dist[:, None, :]) ** (2.0 / (m - 1)), axis=2
        )
        # Converged once memberships stop moving.
        if np.linalg.norm(U - U_old) < tol:
            break
    return centers, U


def _xie_beni(X, centers, U, m=2):
    """Xie-Beni validity index — lower is better."""
    # Undefined for a single cluster (no between-centre distance), so return
    # infinity to make sure c=1 is never selected as the best.
    if len(centers) < 2:
        return np.inf
    dist = np.sum((X[:, None] - centers[None]) ** 2, axis=2)
    # Numerator = total fuzzy within-cluster compactness.
    num = np.sum((U ** m) * dist)
    center_dist = cdist(centers, centers, "sqeuclidean")
    # Diagonal is the distance of a centre to itself (0); set to inf so it is
    # never chosen as the minimum separation.
    np.fill_diagonal(center_dist, np.inf)
    # Divide compactness by the tightest centre pair: low = compact and well separated.
    return num / (X.shape[0] * np.min(center_dist))


def fuzzy_cmeans_auto(X, df, features, c_range=(2, 10), random_state=42):
    """Sweep c_range, pick c with lowest Xie-Beni; return hard labels."""
    best_xb = np.inf
    best_labels = np.zeros(len(X), dtype=int)
    best_c = c_range[0]
    for c in range(*c_range):          # c_range=(2,10) sweeps c = 2..9
        centers, U = _fuzzy_c_means_fixed(X, c, random_state=random_state)
        xb = _xie_beni(X, centers, U)
        if xb < best_xb:               # lower Xie-Beni is better
            best_xb = xb
            # Defuzzify: assign each sample to its highest-membership cluster,
            # so the ensemble downstream receives a hard partition.
            best_labels = np.argmax(U, axis=1)
            best_c = c
    report_clusters(df, best_labels, features, f"FuzzyCMeans (best c={best_c})")
    return best_labels

### 2c · TwoStep  (BIRCH + GMM)

In [ ]:
def two_step_clustering(X, df, features, max_clusters=12,
                        threshold=0.5, random_state=42):
    """
    Step 1 — BIRCH compresses data into subclusters.
    Step 2 — GMM (BIC-selected K) fitted on birch.subcluster_centers_.
    Bug 5 fix: uses subcluster_centers_ (not manual means).
    """
    # BIRCH with n_clusters=None stops after building the CF-tree, so we get
    # the raw subcluster centres rather than a final partition. Fitting the GMM
    # on those few hundred centres instead of all N points is what makes this
    # "two step" approach fast on large data.
    birch = Birch(threshold=threshold, n_clusters=None)
    birch.fit(X)
    subcluster_centers = birch.subcluster_centers_
    # Degenerate guard: with fewer than 2 subclusters there is nothing to model.
    if subcluster_centers is None or len(subcluster_centers) < 2:
        return np.zeros(len(X), dtype=int)
    # Cannot fit more mixture components than there are points to fit them on.
    max_k = min(max_clusters, len(subcluster_centers) - 1)
    if max_k < 2:
        return np.zeros(len(X), dtype=int)
    best_bic, best_gmm = np.inf, None
    for k in range(2, max_k + 1):
        gmm = GaussianMixture(
            n_components=k, covariance_type="full", random_state=random_state
        )
        gmm.fit(subcluster_centers)
        # BIC penalises model complexity, so it can be compared across K
        # directly — unlike log-likelihood, which always improves with more K.
        bic = gmm.bic(subcluster_centers)
        if bic < best_bic:
            best_bic = bic
            best_gmm = gmm
    # Fitted on the compressed centres, but PREDICTED on every original sample.
    final_labels = best_gmm.predict(X)
    report_clusters(df, final_labels, features, "TwoStep")
    return final_labels

### 2d · Spectral, KMeans, GMM, Hierarchical, HDBSCAN

In [ ]:
def spectral_cluster(X, df, features, max_k=10, random_state=42):
    """Auto-select K via Laplacian eigengaps."""
    # Build a k-nearest-neighbour connectivity graph, then its normalised
    # Laplacian. The eigenvalue spectrum of that Laplacian encodes how many
    # well-separated connected components the graph approximately has.
    
    A = kneighbors_graph(X, n_neighbors=10, include_self=False, mode="connectivity")
    L = csgraph.laplacian(A, normed=True)
    
    # which="SM" = smallest magnitude: only the low end of the spectrum carries
    # the cluster-count signal, so a partial solve is enough.
    
    eigvals = np.sort(eigsh(L, k=max_k + 1, which="SM", return_eigenvectors=False))
    gaps = np.diff(eigvals)
    
    # Eigengap heuristic: the largest jump between consecutive eigenvalues
    # indicates the natural number of clusters. max(2, ...) forbids K=1.
    k = max(2, int(np.argmax(gaps)) + 1)
    labels = SpectralClustering(
        n_clusters=k, affinity="nearest_neighbors", random_state=random_state
    ).fit_predict(X)
    report_clusters(df, labels, features, f"Spectral (k={k})")
    return labels


def kmeans_cluster(X, df, features, max_k=12, random_state=42):
    """Auto-select K by maximum silhouette score."""
    best_score, best_labels = -1.0, np.zeros(len(X), dtype=int)
    for k in range(2, min(max_k, X.shape[0] - 1) + 1):
        labels = KMeans(
            n_clusters=k, random_state=random_state, n_init="auto"
        ).fit_predict(X)
        # A collapsed solution has no silhouette; skip rather than error.
        if len(np.unique(labels)) < 2:
            continue
        # Silhouette in [-1, 1]: high = points sit closer to their own cluster
        # than to the nearest neighbouring cluster.
        score = silhouette_score(X, labels)
        if score > best_score:
            best_score = score
            best_labels = labels
    report_clusters(df, best_labels, features, "KMeans")
    return best_labels


def gmm_cluster(X, df, features, max_k=12, random_state=42):
    """Auto-select K by minimum BIC."""
    best_bic, best_labels = np.inf, np.zeros(len(X), dtype=int)
    for k in range(2, min(max_k, X.shape[0] - 1) + 1):  # Bug1-fix: start at 2 (k=1 is degenerate)
        gmm = GaussianMixture(n_components=k, random_state=random_state)
        gmm.fit(X)
        bic = gmm.bic(X)      # lower BIC = better fit after complexity penalty
        if bic < best_bic:
            best_bic = bic
            best_labels = gmm.predict(X)
    report_clusters(df, best_labels, features, "GMM")
    return best_labels


def hierarchical_cluster(X, df, features, max_k=12):
    """Auto-select K by maximum silhouette score (Ward linkage)."""
    best_score, best_labels = -1.0, np.zeros(len(X), dtype=int)
    for k in range(2, min(max_k, X.shape[0] - 1) + 1):
        # Ward minimises within-cluster variance, which pairs naturally with
        # the standardised Euclidean feature space used here.
        labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
        score = silhouette_score(X, labels)
        if score > best_score:
            best_score = score
            best_labels = labels
    report_clusters(df, best_labels, features, "Hierarchical")
    return best_labels


def hdbscan_cluster(X, df, features):
    """Density-based — K determined automatically."""
    # sqrt(N) is a common rule of thumb for the smallest admissible cluster.
    # NOTE: this function is defined but is NOT called in Stage 1 — see the
    # "Known issues" list in the notebook header.
    min_cluster_size = max(5, int(np.sqrt(len(X))))
    labels = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size).fit_predict(X)
    report_clusters(df, labels, features, "HDBSCAN")
    return labels


print("Algorithm definitions loaded.")

## 3 · Ensemble & Ranking Function Definitions

### 3a · Method weights & co-association matrix

In [ ]:
def compute_method_weights(labels_dict, X):
    """
    Softmax-normalised weights = (mean pairwise ARI) + (silhouette).
    Algorithms that collapse to 1 cluster get weight 0.
    """
    # Rationale: an algorithm earns influence in the ensemble by being both
    # (a) consistent with its peers and (b) internally well-formed. Combining
    # the two guards against a lone algorithm with a high silhouette but a
    # partition nobody else reproduces, and vice versa.
    methods = list(labels_dict.keys())
    cons, qual = {}, {}
    for m in methods:
        # A degenerate single-cluster solution is uninformative — zero weight.
        if len(np.unique(labels_dict[m])) < 2:
            cons[m] = 0.0; qual[m] = 0.0
            continue
        # Consensus term: mean ARI against every OTHER algorithm.
        aris = [
            adjusted_rand_score(labels_dict[m], labels_dict[t])
            for t in methods if t != m
        ]
        cons[m] = float(np.mean(aris)) if aris else 0.0
        try:
            # Quality term: silhouette, floored at 0 so a negative silhouette
            # cannot cancel out the consensus term.
            qual[m] = max(0.0, float(silhouette_score(X, labels_dict[m])))
        except Exception:
            qual[m] = 0.0
    scores = np.array([cons[m] + qual[m] for m in methods])
    # If every algorithm scores identically, softmax is undefined in spirit —
    # fall back to equal weighting.
    if scores.max() == scores.min():
        w = np.ones(len(methods)) / len(methods)
    else:
        # Subtracting the max before exponentiating is the standard numerically
        # stable softmax (prevents overflow). NOTE: because the raw scores lie
        # in a narrow range, the resulting weights are close to uniform.
        w = np.exp(scores - scores.max())
        w /= w.sum()
    return dict(zip(methods, w))


def build_coassoc(labels_dict, weights):
    """
    Weighted N×N co-association matrix.

    Opt-fix (build_coassoc): replaced full N×N boolean outer-product with
    cluster-block updates.  For each algorithm and each cluster c, only the
    sub-block C[np.ix_(idx_c, idx_c)] is incremented (+w), avoiding the
    construction of a dense NxN boolean matrix.  Complexity drops from
    O(A·N²) to O(A·Σk n_k²) — roughly 3x faster for K=3 balanced clusters.
    """
    # C[i, j] ends up as the weighted fraction of algorithms that placed
    # samples i and j in the same cluster: an empirical similarity built from
    # agreement rather than from feature-space distance. This is the core
    # object of the whole ensemble.
    n = len(next(iter(labels_dict.values())))
    C = np.zeros((n, n))
    total_weight = 0.0
    for m, labels in labels_dict.items():
        w = weights[m]
        total_weight += w
        for c in np.unique(labels):
            if c == -1:           # skip HDBSCAN noise points
                continue
            idx = np.where(labels == c)[0]
            C[np.ix_(idx, idx)] += w   # only touch the cluster sub-block
    # Normalise so C lies in [0, 1] and is interpretable as a co-clustering
    # probability regardless of the weight scale.
    if total_weight > 0:
        C /= total_weight
    # A sample always co-clusters with itself.
    np.fill_diagonal(C, 1.0)
    return C

### 3b · Optimal K selection

In [ ]:
def ensemble_eigengaps(C, max_k=10, eps=1e-10):
    """Normalised eigengaps of the symmetric normalised Laplacian of C.

    Bug2-fix: replaced full dense eigh() (O(N³)) with eigsh() requesting
    only (max_k+2) smallest eigenvalues — far faster for N≈4000.
    """
    # Treats the co-association matrix as a weighted graph and asks the same
    # question spectral clustering asks: how many near-disconnected components
    # does it contain? This gives an ensemble-level opinion on K that is
    # independent of silhouette or ARI.
    from scipy.sparse import diags as sp_diags
    from scipy.sparse.linalg import eigsh as sp_eigsh

    n = len(C)
    d = C.sum(axis=1)                  # degree of each node
    # eps guards against a zero-degree node producing a divide-by-zero.
    d_inv_sqrt = 1.0 / np.sqrt(d + eps)

    # Build normalised Laplacian as a sparse-friendly computation:
    #   L_sym = I - D^{-1/2} C D^{-1/2}
    # We work with scipy.sparse to avoid materialising dense N×N matrices.
    from scipy.sparse import csr_matrix
    D_inv_sqrt_diag = sp_diags(d_inv_sqrt)
    C_sparse = csr_matrix(C)
    L_sparse = (csr_matrix(np.eye(n))
                - D_inv_sqrt_diag @ C_sparse @ D_inv_sqrt_diag)

    k_req = min(max_k + 2, n - 1)          # eigsh needs k < N
    try:
        eigvals = sp_eigsh(L_sparse, k=k_req, which="SM",
                           return_eigenvectors=False)
        eigvals = np.sort(np.real(eigvals))
    except Exception:
        # Fallback to dense if sparse solver fails (e.g., very small N)
        # (ARPACK can fail to converge on small or near-singular problems.)
        eigvals = np.sort(eigh(C, eigvals_only=True))

    target = max_k + 1
    # Pad if the solver returned fewer eigenvalues than requested, so the
    # downstream index lookup never goes out of bounds.
    if eigvals.size < target:
        pad = np.full(target - eigvals.size,
                      eigvals[-1] if eigvals.size > 0 else 0.0)
        eigvals = np.concatenate([eigvals, pad])
    gaps = np.diff(eigvals[:target])
    # Normalise to [0, 1] so this term is on the same scale as silhouette and
    # ARI when the four criteria are averaged in find_best_k_from_consensus.
    return gaps / gaps.max() if gaps.max() > 0 else np.zeros_like(gaps)


def find_best_k_from_consensus(labels_dict, X, C, weights, k_range):
    """
    Score each K on silhouette, weighted ARI, intra-cluster co-association,
    and spectral eigengap.  Returns (best_score, best_k, scores_dict).
    """
    # Four criteria are combined because each is blind in a different way:
    # silhouette sees only geometry, ARI only peer agreement, co-association
    # only ensemble cohesion, and the eigengap only graph structure.
    best = (-np.inf, None)
    eigengaps = ensemble_eigengaps(C, max(k_range))
    scores = {}
    # Convert co-association (similarity) to a distance, 1 - C, and build ONE
    # hierarchy. Every candidate K is then a different cut of the same tree,
    # which is what makes the K comparison internally consistent — and what
    # the k-sensitivity analysis in Stage 4b later re-uses.
    Z = linkage(squareform(np.clip(1 - C, 0, None)), method="average")
    for k in k_range:
        # fcluster returns 1-based labels; -1 makes them 0-based here purely
        # for the metric calls in this loop.
        labels = fcluster(Z, k, criterion="maxclust") - 1
        if len(np.unique(labels)) < 2:
            continue
        Q_int  = float(silhouette_score(X, labels))          # geometric quality
        # Agreement with the base algorithms, weighted by their credibility.
        Q_agr  = float(np.average(
            [adjusted_rand_score(labels, labels_dict[m]) for m in labels_dict],
            weights=list(weights.values())
        ))
        # Mean co-association among pairs placed in the same cluster: how
        # strongly the ensemble itself supports this grouping.
        same_mask = labels[:, None] == labels[None, :]
        Q_stab = float(C[same_mask].mean()) if same_mask.any() else 0.0
        Q_spec = float(eigengaps[k - 1]) if k - 1 < len(eigengaps) else 0.0
        # Unweighted mean: all four criteria are treated as equally important.
        final_score = (Q_int + Q_agr + Q_stab + Q_spec) / 4.0
        scores[k] = (final_score, Q_int, Q_agr, Q_stab, Q_spec)
        if final_score > best[0]:
            best = (final_score, k)
    return best[0], best[1], scores

### 3c · Stability, consistency & ranking

In [ ]:
def _one_boot(Xb, orig_sub, runner):
    """Single picklable bootstrap replication — used by joblib."""
    try:
        lb = runner(Xb)
        # ARI is invariant to label permutation, so the re-clustered labels do
        # not need to be matched back to the original numbering.
        return float(adjusted_rand_score(orig_sub, lb))
    except Exception:
        # A failed replication returns None and is dropped, rather than
        # aborting the whole stability estimate.
        return None


def _bootstrap_stability(X, algorithm_runner, original_labels,
                         n_clusters, B=30, subsample_frac=0.8,
                         random_state=42, n_jobs=1):
    """
    Bootstrap stability via repeated sub-sampling.

    Opt-fix: joblib parallelisation (prefer=threads avoids pickling issues
    with sklearn estimators).  Set n_jobs=-1 to use all CPU cores.
    Bug 1 fix: n_clusters is an explicit param — no global k leakage.
    """
    # The question being answered: if we had sampled 80% of these participants
    # instead, would this algorithm recover the same partition? High mean ARI
    # across replications = the structure is a property of the data, not of
    # this particular sample.
    n = len(X)
    rng = np.random.RandomState(random_state)
    subsets = []
    for _ in range(B):
        # replace=False: sub-sampling without replacement, so no duplicated
        # rows distort the distance geometry the algorithms rely on.
        idx = rng.choice(n, size=int(subsample_frac * n), replace=False)
        # Cannot ask for more clusters than there are points.
        if len(idx) < n_clusters:
            continue
        subsets.append((X[idx], original_labels[idx]))

    if n_jobs != 1 and len(subsets) > 1:
        results = Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(_one_boot)(Xb, lb, algorithm_runner)
            for Xb, lb in subsets
        )
    else:
        results = [_one_boot(Xb, lb, algorithm_runner) for Xb, lb in subsets]

    scores = [r for r in results if r is not None]
    return float(np.mean(scores)) if scores else 0.0


def _cluster_consistency(A, B):
    """
    Bug 6 fix: symmetric (forward + backward) / 2 mean max-Jaccard.
    """
    # Complements ARI: instead of scoring the partition as a whole, it asks
    # whether each individual cluster has a close counterpart in the other
    # solution. Averaging both directions makes it symmetric — otherwise a
    # solution with many small clusters could score highly one way only.
    def _one_dir(X_labs, Y_labs):
        # Represent each cluster as the set of sample indices it contains.
        X_sets = {x: set(np.where(X_labs == x)[0]) for x in np.unique(X_labs)}
        Y_sets = {y: set(np.where(Y_labs == y)[0]) for y in np.unique(Y_labs)}
        row_scores = []
        for xs in X_sets.values():
            best_j = 0.0
            # Best-matching counterpart cluster by Jaccard overlap.
            for ys in Y_sets.values():
                u = len(xs | ys)
                if u > 0:
                    j = len(xs & ys) / u
                    if j > best_j:
                        best_j = j
            row_scores.append(best_j)
        return float(np.mean(row_scores)) if row_scores else 0.0
    return (_one_dir(A, B) + _one_dir(B, A)) / 2.0


def _get_runners(k, random_state=42):
    """
    Fixed-K runners mirroring Stage-4 exactly.
    Bug 2 fix: SOM removed — 6 keys match new_label_dict.
    Bug 5 fix: run_twostep uses subcluster_centers_.
    """
    # These closures must reproduce Stage 4 exactly, otherwise the bootstrap
    # would be measuring the stability of a DIFFERENT algorithm than the one
    # being ranked. Each takes a data subset and returns labels at fixed k.
    def run_fuzzy(X_subset):
        n = X_subset.shape[0]
        rng = np.random.default_rng(random_state)
        # Seeded membership init so replications are reproducible.
        init_u = rng.random((k, n))
        init_u /= init_u.sum(axis=0, keepdims=True)
        try:
            # skfuzzy expects features-by-samples, hence the transpose.
            _, u, *_ = fuzz.cluster.cmeans(
                X_subset.T, c=k, m=2, error=1e-5, maxiter=350, init=init_u
            )
            return np.argmax(u, axis=0)   # defuzzify to hard labels
        except Exception:
            return np.zeros(n, dtype=int)

    def run_twostep(X_subset):
        try:
            birch = Birch(n_clusters=None)
            birch.fit(X_subset)
            subcenters = birch.subcluster_centers_
            # If BIRCH produced fewer centres than k, fall back to fitting the
            # GMM directly on the subset rather than failing.
            n_comp = (
                k if (subcenters is not None and len(subcenters) >= k)
                else max(1, len(subcenters) if subcenters is not None else 1)
            )
            gmm = GaussianMixture(n_components=n_comp, random_state=random_state)
            if subcenters is not None and len(subcenters) >= k:
                gmm.fit(subcenters)          # fit on compressed centres
                return gmm.predict(X_subset) # predict on all samples
            return gmm.fit_predict(X_subset)
        except Exception:
            return np.zeros(X_subset.shape[0], dtype=int)

    # Keys MUST match new_label_dict in Stage 4, or rank_algorithms silently
    # skips an algorithm.
    return {
        "kmeans":        lambda X: KMeans(n_clusters=k, random_state=random_state, n_init="auto").fit_predict(X),
        "gmm":           lambda X: GaussianMixture(n_components=k, random_state=random_state).fit_predict(X),
        "agglomerative": lambda X: AgglomerativeClustering(n_clusters=k).fit_predict(X),
        "spectral":      lambda X: SpectralClustering(n_clusters=k, affinity="nearest_neighbors", random_state=random_state).fit_predict(X),
        "fuzzycmeans":   run_fuzzy,
        "twostep":       run_twostep,
    }


def rank_algorithms(labels_dict, X, consensus_labels, consensus_k, random_state=42):
    """
    Score each algorithm on silhouette, ARI vs consensus, peer consistency,
    and bootstrap stability. Returns a DataFrame sorted by FinalScore.
    """
    # The four axes are deliberately different in kind: internal geometry,
    # agreement with the ensemble, agreement with peers cluster-by-cluster,
    # and robustness to resampling. An algorithm has to do well on all four
    # to rank highly.
    method_runners = _get_runners(consensus_k, random_state)
    rows = []
    print(f"  Ranking {len(labels_dict)} algorithms at K = {consensus_k} ...")
    for m, labels in labels_dict.items():
        if m not in method_runners:
            print(f"  Warning: no runner for {m!r} — skipping.")
            continue
        n_uniq = len(np.unique(labels))
        # Guard every metric against a degenerate single-cluster partition.
        sil  = float(silhouette_score(X, labels))          if n_uniq > 1 else 0.0
        ch   = float(calinski_harabasz_score(X, labels))   if n_uniq > 1 else 0.0
        # DB is a "lower is better" index, so its failure value is +inf.
        db   = float(davies_bouldin_score(X, labels))      if n_uniq > 1 else np.inf
        ari  = float(adjusted_rand_score(labels, consensus_labels))
        others = [labels_dict[t] for t in labels_dict if t != m]
        cc   = float(np.mean([_cluster_consistency(labels, o) for o in others])) if others else 0.0
        stability = _bootstrap_stability(
            X, method_runners[m], labels,
            n_clusters=consensus_k, B=30, subsample_frac=0.8,
            random_state=random_state, n_jobs=1   # set n_jobs=-1 for parallelism
        )
        # NOTE: CH and DB are reported for information but are NOT part of
        # FinalScore — their scales are unbounded and would dominate the mean.
        final = float(np.mean([sil, ari, cc, stability]))
        rows.append({
            "Algorithm":      m,
            "FinalScore":     round(final,      4),
            "Silhouette":     round(sil,         4),
            "CH_Index":       round(ch,          2),
            "DB_Index":       round(db,          4),
            "ARI_Consensus":  round(ari,         4),
            "Consistency":    round(cc,          4),
            "Stability":      round(stability,   4),
        })
    df_rank = pd.DataFrame(
        rows,
        columns=["Algorithm", "FinalScore", "Silhouette",
                 "CH_Index", "DB_Index",
                 "ARI_Consensus", "Consistency", "Stability"]
    )
    return df_rank.sort_values("FinalScore", ascending=False).reset_index(drop=True)


print("Ensemble & ranking definitions loaded.")

## 4 · Load & Preprocess Data

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# UPSTREAM DEPENDENCY CHECK
# ══════════════════════════════════════════════════════════════════════════
# For WTMEC2YR (NHANES survey weight) to be present in the output CSV,
# t2d_fast_clean.csv must have been generated from the corrected pipeline:
#
#   Step 1 → Run: dataset_merging_new_weighted.ipynb
#              Produces: nhanes_1999_2018_combined_new.csv (with WTMEC2YR)
#
#   Step 2 → Run: EDA_enhanced.ipynb
#              Produces: t2d_fast_clean.csv (WTMEC2YR passes through
#              all row-filtering steps automatically — no code change needed)
#
#   Step 3 → Run this notebook (no changes needed here)
#              WTMEC2YR is carried through via df_export = df.copy()
# ══════════════════════════════════════════════════════════════════════════

df = pd.read_csv("t2d_fast_clean.csv")

# The five clustering features, matching the Ahlqvist-style variable set:
# age, adiposity (BMI), glycaemia (HbA1c), insulin resistance and beta-cell
# function. All five are complete by construction in t2d_fast_clean.
features = ["Age", "BMI", "HbA1c", "HOMA_IR", "HOMA_B"]
X = df[features].copy()

# Log-transform right-skewed metabolic markers
# Without this, a handful of extreme HOMA values would dominate the Euclidean
# distances that every algorithm below depends on.
X["HOMA_IR"] = np.log(X["HOMA_IR"])
X["HOMA_B"]  = np.log(X["HOMA_B"])

# Standardise to mean 0 / SD 1 so that features measured on different units
# (years vs kg/m² vs %) contribute equally to distance. This is essential for
# distance-based clustering — without it, Age would dominate purely by range.
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Single seed reused by every stochastic algorithm below, so the whole
# notebook is reproducible from one constant.
SEED = 42

print(f"Dataset loaded : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Features used  : {features}")
df.head()

# ── Verify WTMEC2YR is present in the loaded data ────────────────────────
# Survey weights are not used for clustering itself, but they must survive
# into the exported CSV for the weighted sensitivity analysis downstream.
if 'WTMEC2YR' in df.columns:
    missing_wt = df['WTMEC2YR'].isna().sum()
    print(f'✅ WTMEC2YR present — {len(df) - missing_wt:,} valid weights '
          f'({missing_wt} missing)')
    print(f'   Weight range: {df["WTMEC2YR"].min():.1f} – '
          f'{df["WTMEC2YR"].max():,.1f}')
else:
    print('⚠️  WTMEC2YR NOT found in t2d_fast_clean.csv')
    print('   → Re-run dataset_merging_new_weighted.ipynb then EDA_enhanced.ipynb first.')
    print('   → Survey-weighted sensitivity analysis will not be possible until resolved.')

## 5 · Stage 1 — Individual Algorithm Optimisation
Each algorithm independently finds its best K using its own internal criterion.

In [ ]:
print("=" * 60)
print("STAGE 1: Individual algorithm optimisation")
print("=" * 60)

# Each algorithm chooses its OWN K by its own internal criterion (silhouette,
# BIC, Xie-Beni or eigengap). Letting them disagree is the point: the spread of
# their answers is the raw material the consensus stage works from.
# NOTE: hdbscan_cluster is defined but deliberately not included here.
labels_dict = {
    "TwoStep":      two_step_clustering(X_scaled, df, features, random_state=SEED),
    "Spectral":     spectral_cluster   (X_scaled, df, features, random_state=SEED),
    "FuzzyCMeans":  fuzzy_cmeans_auto  (X_scaled, df, features, random_state=SEED),
    "KMeans":       kmeans_cluster     (X_scaled, df, features, random_state=SEED),
    "GMM":          gmm_cluster        (X_scaled, df, features, random_state=SEED),
    "Hierarchical": hierarchical_cluster(X_scaled, df, features),
}

### 5a · Cluster counts & medians per algorithm

In [ ]:
# Inspect each algorithm's solution on the ORIGINAL clinical scale, not the
# standardised one — medians in years, kg/m² and % are what make a cluster
# clinically recognisable as SIDD / MARD / SOIRD.
for name, labels in labels_dict.items():
    temp_df = df.copy()
    temp_df["Cluster"] = labels
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels[unique_labels != -1])   # ignore noise label
    print(f"\n--- {name}  ({n_clusters} clusters) ---")
    print("Counts:")
    print(temp_df["Cluster"].value_counts().sort_index().to_string())
    print("\nMedians (original scale):")
    # Median rather than mean: robust to the residual skew in HOMA indices.
    print(temp_df.groupby("Cluster")[features].median().to_string())
    print("-" * 40)

## 6 · Stage 2 — Weighted Consensus Matrix
Algorithms are weighted by (mean peer ARI + silhouette) via softmax,
then combined into a weighted co-association matrix **C**.

In [ ]:
print("=" * 60)
print("STAGE 2: Building weighted consensus matrix")
print("=" * 60)

# Credibility weight per algorithm (peer agreement + internal quality).
weights = compute_method_weights(labels_dict, X_scaled)

print("\nAlgorithm weights:")
for m, w in weights.items():
    print(f"  {m:<14}  {w:.4f}")

# C[i, j] = weighted proportion of algorithms that co-clustered samples i and j.
C = build_coassoc(labels_dict, weights)
print(f"\nCo-association matrix shape : {C.shape}")
# Range check: values must lie in [0, 1]. A max below 1 outside the diagonal
# would mean no pair was placed together by every algorithm.
print(f"Value range                 : [{C.min():.3f}, {C.max():.3f}]")

## 7 · Stage 3 — Optimal K Selection
Each candidate K is scored on four axes: silhouette, weighted ARI,
intra-cluster co-association, and normalised spectral eigengap.

In [ ]:
print("=" * 60)
print("STAGE 3: Determining optimal global K")
print("=" * 60)

# Sweep K = 2..9 and score each on the four criteria. This is the decision
# that fixes the number of phenotypes reported in the manuscript.
best_score, best_k, k_scores = find_best_k_from_consensus(
    labels_dict, X_scaled, C, weights, k_range=range(2, 10)
)

print(f"\n{'K':<4} | {'Score':<8} | {'Sil':<8} | {'ARI':<8} | {'Stab':<8} | {'Eigengap':<8}")
print("-" * 65)
# Printing every K, not just the winner, so a reader can see how close the
# runner-up was — the transparency a reviewer will look for.
for k, vals in sorted(k_scores.items()):
    marker = "  <-- best" if k == best_k else ""
    print(
        f"{k:<4} | {vals[0]:.4f}   | {vals[1]:.3f}    | "
        f"{vals[2]:.3f}    | {vals[3]:.3f}    | {vals[4]:.3f}{marker}"
    )

print(f"\nOptimal consensus K = {best_k}  (score = {best_score:.4f})")

# Rebuild the SAME linkage used inside find_best_k_from_consensus so the final
# partition is exactly the winning cut of that tree. Z_cons is retained because
# Stage 4b re-cuts it at other K values for the sensitivity analysis.
Z_cons           = linkage(squareform(np.clip(1 - C, 0, None)), method="average")
# fcluster returns 1-based labels (1, 2, 3) — keep them 1-based so the
# exported Consensus_Cluster column and every downstream plot use 1/2/3
# rather than 0/1/2.  This is the single source of truth for the label
# base; no other cell shifts these values.
consensus_labels = fcluster(Z_cons, best_k, criterion="maxclust")
print(f"Consensus labels (first 20): {consensus_labels[:20]}")
print(f"Unique consensus labels    : {sorted(set(consensus_labels))}")

## 8 · Stage 4 — Fixed-K Re-run at Consensus K
All 6 algorithms are re-run at `best_k` to produce comparable label sets for ranking.

In [ ]:
print("=" * 60)
print(f"STAGE 4: Fixed-K re-run at K = {best_k}")
print("=" * 60)

# Why re-run: in Stage 1 each algorithm used its own K, so their partitions
# are not directly comparable. Forcing every algorithm to the consensus K
# yields label sets that CAN be compared like-for-like in Stage 5.
k = best_k

labels_K = KMeans(
    n_clusters=k, random_state=SEED, n_init="auto"
).fit_predict(X_scaled)

labels_G = GaussianMixture(
    n_components=k, random_state=SEED
).fit_predict(X_scaled)

labels_A = AgglomerativeClustering(
    n_clusters=k
).fit_predict(X_scaled)

labels_S = SpectralClustering(
    n_clusters=k, affinity="nearest_neighbors", random_state=SEED
).fit_predict(X_scaled)

# Fuzzy C-Means via skfuzzy (seeded init)
# NOTE: this is skfuzzy's implementation, whereas Stage 1 used the local
# pure-numpy _fuzzy_c_means_fixed — see "Known issues" in the header.
rng    = np.random.default_rng(SEED)
init_u = rng.random((k, X_scaled.shape[0]))
init_u /= init_u.sum(axis=0, keepdims=True)   # normalise memberships per sample
_, u, *_ = fuzz.cluster.cmeans(
    X_scaled.T, c=k, m=2, error=1e-5, maxiter=350, init=init_u
)
labels_F = np.argmax(u, axis=0)               # defuzzify

# TwoStep — Bug 5 fix: subcluster_centers_ (consistent with Stage 1)
birch = Birch(n_clusters=None)
birch.fit(X_scaled)
subcenters = birch.subcluster_centers_
if subcenters is not None and len(subcenters) >= k:
    gmm_ts = GaussianMixture(n_components=k, random_state=SEED)
    gmm_ts.fit(subcenters)                    # fit on compressed centres
    labels_T = gmm_ts.predict(X_scaled)       # assign every sample
else:
    # Fallback when BIRCH yields too few centres to support k components.
    labels_T = GaussianMixture(n_components=k, random_state=SEED).fit_predict(X_scaled)

# ── Shift all 0-based sklearn labels to 1-based (1…K) so that ──────────
# cluster numbering matches the consensus (fcluster returns 1-based) and
# CLUSTER_PALETTE colours are applied uniformly across all algorithms.
# This is cosmetic only: it changes label VALUES, never the partition, and
# ARI-based comparisons are unaffected because ARI ignores label identity.
labels_K = labels_K + 1
labels_G = labels_G + 1
labels_A = labels_A + 1
labels_S = labels_S + 1
labels_F = labels_F + 1
labels_T = labels_T + 1

# Keys here MUST match those returned by _get_runners(), or rank_algorithms
# will skip the mismatched algorithm with a warning.
new_label_dict = {
    "kmeans":        labels_K,
    "gmm":           labels_G,
    "agglomerative": labels_A,
    "spectral":      labels_S,
    "fuzzycmeans":   labels_F,
    "twostep":       labels_T,
}

print("\nCluster counts per algorithm:")
print("=" * 36)
# Sizes are the quickest diagnostic: an algorithm producing one dominant
# cluster plus slivers has effectively failed even though K matches.
for name, lab in new_label_dict.items():
    counts = pd.Series(lab).value_counts().sort_index()
    print(f"\n{name}  (unique = {len(np.unique(lab))})")
    print(counts.to_string())
    print("-" * 36)

## 8b · Stage 4b — k-Sensitivity Analysis

Tests whether the published k=3 partition is empirically supported, or whether higher granularity (k=4, k=5) reveals separable subtypes that the composite score collapsed.

**Specific question for the manuscript:** does the SOIRD cluster at k=3 split into separable MOD-like and SIRD-like phenotypes when k=4 is forced?

**Approach:** the consensus dendrogram `Z_cons` already encodes the full hierarchical structure derived from the weighted co-association matrix `C`. To obtain the consensus partition at any k, we re-cut `Z_cons` at that height. This is exact (no re-sampling, no re-running base algorithms) and uses the same consensus information that produced the published k=3 result.

Outputs are written to `k_sensitivity_outputs/` for inclusion as Supplementary Table S2 and Supplementary Figure S1.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Stage 4b — Cell 1: Re-cluster at alternative k by re-cutting Z_cons
# ══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STAGE 4b: k-Sensitivity Analysis")
print("=" * 60)

# Cut the existing consensus dendrogram at alternative k values.
# Z_cons was built in Stage 3 as: linkage(squareform(np.clip(1 - C, 0, None)), method='average')
# Re-cutting the SAME tree (rather than re-running the ensemble at each k) is
# what makes these partitions directly nested and comparable: every k here
# derives from identical consensus information.
ks_to_test = [2, 3, 4, 5]
sensitivity_labels = {}
for k_val in ks_to_test:
    sensitivity_labels[k_val] = fcluster(Z_cons, k_val, criterion="maxclust")

# Sanity check: k=3 from Z_cons should match published consensus_labels exactly
# If this fails, Z_cons has been rebuilt from a different C and none of the
# sensitivity results would be comparable to the published partition.
# NOTE: this assertion assumes best_k == 3; it will fail if K selection changes.
assert np.array_equal(sensitivity_labels[3], consensus_labels), (
    "Re-cut at k=3 does not match published consensus_labels — check Z_cons integrity"
)
print("✓ Sanity check passed: re-cut k=3 matches published consensus_labels\n")

# Composite scores from Stage 3 (already computed in k_scores dict)
# Reprinted here so the supplement can show how close competing k values were.
print("Composite scores from Stage 3 cluster-number selection:")
print(f"{'K':<4} | {'Composite':<10} | {'Silhouette':<11} | {'Weighted ARI':<13} | "
      f"{'Stability':<10} | {'Eigengap':<9}")
print("-" * 75)
for k_val in sorted(k_scores.keys()):
    vals = k_scores[k_val]
    marker = "  <-- selected" if k_val == best_k else ""
    print(f"{k_val:<4} | {vals[0]:<10.4f} | {vals[1]:<11.3f} | {vals[2]:<13.3f} | "
          f"{vals[3]:<10.3f} | {vals[4]:<9.3f}{marker}")

# Cluster sizes summary
# The min/max ratio is a balance diagnostic: a value near 0 means the extra
# cluster at that k is a sliver rather than a substantive phenotype.
print("\nCluster sizes across k:")
print(f"{'K':<4} | {'Sizes (cluster:n)':<50} | {'min/max':<10}")
print("-" * 75)
for k_val in ks_to_test:
    sizes = pd.Series(sensitivity_labels[k_val]).value_counts().sort_index()
    sizes_str = "  ".join(f"{c}:{n}" for c, n in sizes.items())
    ratio = sizes.min() / sizes.max()
    print(f"{k_val:<4} | {sizes_str:<50} | {ratio:<10.3f}")

# Cross-tabulation: where does each k=3 cluster go in k=4?
# This is the core evidence for the manuscript's MOD-merger argument: it shows
# whether the extra cluster at k=4 comes from splitting one k=3 phenotype.
print("\n" + "=" * 60)
print("Cross-tabulation: k=3 (rows) vs k=4 (columns) — RAW COUNTS")
print("=" * 60)
crosstab_counts = pd.crosstab(
    pd.Series(sensitivity_labels[3], name='k=3'),
    pd.Series(sensitivity_labels[4], name='k=4'),
    margins=True, margins_name='Total'
)
print(crosstab_counts)

print("\n" + "=" * 60)
print("Cross-tabulation: ROW % (where each k=3 cluster goes in k=4)")
print("=" * 60)
# normalize='index' makes each ROW sum to 100%, answering "where did this
# k=3 cluster's members end up?" rather than "who makes up this k=4 cluster?".
crosstab_pct = pd.crosstab(
    pd.Series(sensitivity_labels[3], name='k=3'),
    pd.Series(sensitivity_labels[4], name='k=4'),
    normalize='index'
) * 100
print(crosstab_pct.round(1))

print("\nINTERPRETATION GUIDE:")
print("  • If a single k=4 column receives >=85% of a k=3 cluster's members,")
print("    that k=3 cluster did NOT split substantively at higher granularity.")
print("  • If a k=3 cluster splits 60/40 or more evenly across two k=4 columns,")
print("    that is a substantive split — examine the feature profiles in Cell 3.")
print("  • The SOIRD row is the critical one for the MOD-merger argument.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Stage 4b — Cell 2: Feature profiles at k=4 (does SOIRD split into MOD+SIRD?)
# ══════════════════════════════════════════════════════════════════════════
labels_k4 = sensitivity_labels[4]
labels_k3 = sensitivity_labels[3]

# ── Raw medians at k=4 (clinical interpretability) ────────────────────────
# Original units first: this is how a clinician judges whether a cluster is a
# recognisable phenotype rather than a statistical artefact.
print("Raw feature medians at k=4 (original clinical units):")
print("=" * 60)
profile_raw = df[features].copy()
profile_raw['k4'] = labels_k4
medians_k4 = profile_raw.groupby('k4').median().round(2)
medians_k4['n'] = profile_raw['k4'].value_counts().sort_index()
medians_k4['pct'] = (100 * medians_k4['n'] / len(profile_raw)).round(1)
print(medians_k4)

# ── Z-score profiles using the standardised X used for clustering ─────────
# (matches the manuscript Figure 2a methodology exactly)
# Z-scores show each cluster's position RELATIVE to the cohort mean, which is
# what makes profiles comparable across features with different units.
print("\n" + "=" * 60)
print("Z-score profiles at k=4 (log-transformed + standardised features):")
print("=" * 60)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
zscore_k4 = X_scaled_df.groupby(labels_k4).mean().round(2)
print(zscore_k4)

# ── Reference: the published k=3 profile for direct comparison ────────────
print("\n" + "=" * 60)
print("REFERENCE — Z-score profiles at k=3 (the published partition):")
print("=" * 60)
zscore_k3 = X_scaled_df.groupby(labels_k3).mean().round(2)
print(zscore_k3)

# Manuscript labelling reference:
#   k=3 Cluster 1 (n≈527)   → SIDD   : high HbA1c, low HOMA-B, youngest
#   k=3 Cluster 2 (n≈1850)  → MARD   : oldest, otherwise mild
#   k=3 Cluster 3 (n≈1544)  → SOIRD  : highest BMI + HOMA-IR + HOMA-B
# CAUTION: these n values and the cluster→phenotype mapping are hard-coded
# from one run. fcluster numbering is not guaranteed stable if C changes, so
# re-verify the mapping against the printed profiles above after any re-run.
print("\n" + "=" * 60)
print("DIAGNOSTIC: which k=3 phenotype does each k=4 cluster derive from?")
print("=" * 60)
# For each k=4 cluster, find the dominant k=3 cluster of origin
# A k=3 cluster appearing as the origin of TWO k=4 clusters is direct evidence
# that it split.
origin_table = []
for k4_id in sorted(np.unique(labels_k4)):
    members = np.where(labels_k4 == k4_id)[0]
    k3_origin = pd.Series(labels_k3[members]).value_counts(normalize=True) * 100
    dominant = k3_origin.idxmax()
    pct_dominant = k3_origin.max()
    origin_table.append({
        'k4_cluster': k4_id,
        'n': len(members),
        'dominant_k3_origin': int(dominant),
        'pct_from_dominant': round(pct_dominant, 1),
    })
origin_df = pd.DataFrame(origin_table)
print(origin_df.to_string(index=False))

print("\nINTERPRETATION:")
print("  • If two k=4 clusters both derive from the same k=3 origin (especially k=3 Cluster 3 / SOIRD),")
print("    that k=3 cluster has split. Compare their Z-score profiles above:")
print("      → If one has higher BMI / lower HOMA-IR (MOD-like) and the other")
print("        has lower BMI / higher HOMA-IR (SIRD-like) → MOD/SIRD recovered (Template B)")
print("      → If the split is along a different axis (e.g., age) → not a MOD/SIRD recovery (Template A)")
print("  • If each k=4 cluster derives mostly (>=85%) from a single k=3 origin,")
print("    examine which k=3 cluster (if any) was split.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Stage 4b — Cell 3: Supplementary visualisation + export
# ══════════════════════════════════════════════════════════════════════════
import os
import seaborn as sns
import matplotlib.pyplot as plt

SENSITIVITY_DIR = "k_sensitivity_outputs"
os.makedirs(SENSITIVITY_DIR, exist_ok=True)   # exist_ok so re-runs do not error

# ── Side-by-side heatmap: k=3 (left) vs k=4 (right), Z-scores ─────────────
# width_ratios [3, 4] gives each panel width proportional to its cluster count,
# so the individual cells stay the same size across the two heatmaps.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5),
                          gridspec_kw={'width_ratios': [3, 4]})

# center=0 with a diverging palette means red/blue read directly as
# above/below the cohort mean; the shared vmin/vmax makes the two panels
# directly comparable rather than each self-scaling.
sns.heatmap(zscore_k3, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1.5, vmax=1.5, cbar=False, ax=axes[0],
            linewidths=0.5, linecolor='white')
axes[0].set_title("Published partition — k=3\n(SIDD / MARD / SOIRD)", fontsize=11)
axes[0].set_xlabel("")
axes[0].set_ylabel("Cluster")

sns.heatmap(zscore_k4, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1.5, vmax=1.5, cbar_kws={'label': 'Z-score'}, ax=axes[1],
            linewidths=0.5, linecolor='white')
axes[1].set_title("Sensitivity partition — k=4", fontsize=11)
axes[1].set_xlabel("")
axes[1].set_ylabel("Cluster")

plt.suptitle("Supplementary Figure S1 — k-sensitivity comparison of cluster feature profiles",
             fontsize=12, y=1.02)
plt.tight_layout()

# Save vector (PDF, for typesetting) and raster (PNG, for preview) together.
fig_base = os.path.join(SENSITIVITY_DIR, "supp_fig_S1_k_sensitivity_profiles")
for ext in ("pdf", "png"):
    plt.savefig(f"{fig_base}.{ext}", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_base}.pdf and .png")

# ── Export all sensitivity tables as CSV for the supplement ───────────────
# Each file maps to a numbered supplementary table so the manuscript can cite
# them directly without manual re-formatting.
crosstab_counts.to_csv(os.path.join(SENSITIVITY_DIR, "supp_table_S2a_crosstab_k3_vs_k4_counts.csv"))
crosstab_pct.round(1).to_csv(os.path.join(SENSITIVITY_DIR, "supp_table_S2b_crosstab_k3_vs_k4_rowpct.csv"))
medians_k4.to_csv(os.path.join(SENSITIVITY_DIR, "supp_table_S2c_k4_raw_medians.csv"))
zscore_k4.to_csv(os.path.join(SENSITIVITY_DIR, "supp_table_S2d_k4_zscores.csv"))
origin_df.to_csv(os.path.join(SENSITIVITY_DIR, "supp_table_S2e_k4_origin_diagnostic.csv"), index=False)

# ── Composite-score table for the supplement ──────────────────────────────
# Same numbers printed in Cell 1, reshaped into a citable table with an
# explicit flag marking which k was selected.
score_rows = []
for k_val in sorted(k_scores.keys()):
    vals = k_scores[k_val]
    score_rows.append({
        'K': k_val,
        'Composite_Score': round(vals[0], 4),
        'Silhouette': round(vals[1], 3),
        'Weighted_ARI': round(vals[2], 3),
        'Co_assoc_Stability': round(vals[3], 3),
        'Eigengap': round(vals[4], 3),
        'Selected': 'Yes' if k_val == best_k else ''
    })
score_table = pd.DataFrame(score_rows)
score_table.to_csv(os.path.join(SENSITIVITY_DIR, "supp_table_S2f_composite_scores.csv"), index=False)
print("\nSupplementary Table S2f — Composite scores across k:")
print(score_table.to_string(index=False))

# ── Attach sensitivity labels to df for downstream survival sensitivity ───
# These columns will be picked up by df_export = df.copy() in Cell 69,
# making them available to the survival analysis notebook for Cox sensitivity
# at k=4 if the SOIRD-splits-into-MOD/SIRD scenario (Template B) applies.
# NOTE: this MUTATES df, so the exported CSV gains four extra columns.
for k_val in ks_to_test:
    df[f'Sensitivity_k{k_val}'] = sensitivity_labels[k_val]

print(f"\n✅ k-sensitivity analysis complete.")
print(f"   Outputs: {SENSITIVITY_DIR}/")
print(f"   Sensitivity labels added to df as 'Sensitivity_k2', 'Sensitivity_k3',")
print(f"   'Sensitivity_k4', 'Sensitivity_k5' columns (carried through to df_export in Cell 11g).")

## 9 · Stage 5 — Algorithm Ranking

Each algorithm is scored on four axes:
- **Silhouette** — internal compactness/separation
- **ARI_Consensus** — agreement with the ensemble consensus
- **Consistency** — symmetric peer Jaccard consistency
- **Stability** — bootstrap sub-sample ARI (B=30, 80% sub-sample)

In [ ]:
print("=" * 60)
print("STAGE 5: Ranking algorithms")
print("=" * 60)

# Ranks the fixed-K partitions (not the Stage-1 auto-K ones) so all algorithms
# are judged on equal terms. The winner is used for the comparison figure and
# tells us which single algorithm best reproduces the ensemble result.
ranking_df = rank_algorithms(
    new_label_dict,
    X_scaled,
    consensus_labels,
    consensus_k=best_k,
    random_state=SEED,
)

print("\nFinal ranking:")
ranking_df   # bare expression so Jupyter renders the DataFrame as a table

### 5b · Extended internal validation (CH + DB indices)

In [ ]:
## ── Extended Internal Validation Indices ──────────────────────────────────────
## Computed on Stage-4 fixed-K labels + consensus.
## Calinski-Harabasz (CH) — higher is better (ratio of between/within scatter).
## Davies-Bouldin (DB)    — lower is better (average cluster similarity).
## These complement silhouette which can be slow to compute.

def compute_extended_metrics(labels_dict, X, extra_label="Consensus",
                              extra_labels=None):
    """
    Compute CH and DB indices for every algorithm (and an optional extra
    partition such as the consensus solution).

    Returns a tidy DataFrame sorted by Silhouette descending.
    """
    # Three indices are reported together because each can be fooled alone:
    # CH favours many compact clusters, DB penalises overlapping ones, and
    # silhouette is per-sample. Agreement across all three is the strong signal.
    rows = []
    items = list(labels_dict.items())
    # Appending the consensus lets it be judged on exactly the same footing
    # as the individual algorithms.
    if extra_labels is not None:
        items.append((extra_label, extra_labels))

    for name, labels in items:
        n_uniq = len(np.unique(labels[labels != -1]))
        # Degenerate partitions get NaN rather than a misleading number.
        if n_uniq < 2:
            rows.append({"Algorithm": name,
                         "Silhouette": np.nan, "CH_Index": np.nan,
                         "DB_Index": np.nan, "N_Clusters": n_uniq})
            continue

        # Exclude noise points for metric computation
        # Noise is not a cluster, so including it would distort every index.
        mask = labels != -1
        X_m, L_m = X[mask], labels[mask]

        sil = float(silhouette_score(X_m, L_m))
        ch  = float(calinski_harabasz_score(X_m, L_m))
        db  = float(davies_bouldin_score(X_m, L_m))

        rows.append({
            "Algorithm": name,
            "N_Clusters": n_uniq,
            "Silhouette": round(sil, 4),
            "CH_Index":   round(ch,  2),
            "DB_Index":   round(db,  4),
        })

    return (pd.DataFrame(rows)
              .set_index("Algorithm")
              .sort_values("Silhouette", ascending=False))


print("Computing extended internal validation metrics ...")
ext_metrics_df = compute_extended_metrics(
    new_label_dict, X_scaled,
    extra_label="Consensus", extra_labels=consensus_labels
)
print("\nExtended metrics (Stage-4 algorithms + Consensus):")
print(ext_metrics_df.to_string())
print("\nNote: CH higher = better  |  DB lower = better  |  Silhouette higher = better")

## 10 · Visualisation Functions

Five plots covering every key diagnostic view:

| Function | What it shows |
|---|---|
| `plot_coassoc` | Weighted co-association heatmap with consensus partition overlay |
| `plot_ari_matrix` | Pairwise ARI agreement between all algorithms |
| `plot_embedding` | t-SNE 2-D projection coloured by any label vector |
| `plot_all_embeddings` | One t-SNE panel per algorithm + consensus in a grid |
| `plot_cluster_profiles` | Radar / parallel-coordinates of feature medians per cluster |

### 10a · Visualisation imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
import os
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score

# ── Colour maps ───────────────────────────────────────────────────────────────
CMAP_CLUSTERS = "tab10"

# ── Explicit per-cluster palette — guarantees visual consistency across ──────
# ALL figures (t-SNE, UMAP, profiles, boxplots, co-association heatmap).
# ALL Stage-4 algorithm labels are shifted to 1-based (1…K) in Cell 31,
# so this palette applies uniformly to every algorithm AND the consensus.
#
# Colours are chosen for maximum perceptual separation:
#   Cluster 1 → red   (#E64B35)
#   Cluster 2 → blue  (#0000FF)
#   Cluster 3 → green (#00A087)
# IMPORTANT: every key below MUST be unique.
# Fixing colour by cluster ID (rather than by plot order) is what stops
# "cluster 1" being red in one figure and blue in the next.
CLUSTER_PALETTE = {
    1: "#E64B35",   # SIDD  — red
    2: "#0000FF",   # MARD  — blue
    3: "#00A087",   # SOIRD — green
   -1: "#888888",   # noise (grey, if any)
}

def _cluster_color(label, n_clusters=None):
    """Return the canonical RGB hex for a cluster label.

    Strategy:
      • Labels 1, 2, 3 (all algorithms + consensus, 1-based) → fixed red / blue / green.
      • Label −1 (noise/HDBSCAN)                             → grey.
      • Any unexpected label                                 → tab10 colormap, cycled.
    All Stage-4 algorithm labels are shifted to 1-based so colours are
    consistent across every algorithm panel and the consensus.
    """
    # Coerce numpy integer types to Python int so dict lookup works
    # (np.int64(1) != 1 as a dict key, which would silently miss the palette).
    try:
        label = int(label)
    except (TypeError, ValueError):
        pass
    if label in CLUSTER_PALETTE:
        return CLUSTER_PALETTE[label]
    # Fallback for K > 3: cycle tab10 so the function never raises.
    cmap = plt.colormaps[CMAP_CLUSTERS]
    return cmap((label % 10) / 9.0)

CMAP_HEATMAP  = "viridis"     # perceptually uniform, colour-blind safe
CMAP_ARI      = "coolwarm"    # diverging, suits ARI's signed range

# ── Publication-quality global rcParams ───────────────────────────────────────
# Targets journal standards: 300 DPI raster / vector PDF/SVG, Arial/Helvetica
# font, no right/top spines, minimum ink.
PUB_RC = {
    # Font
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "legend.fontsize":    9,
    "legend.title_fontsize": 9,
    # Lines & markers
    "lines.linewidth":    1.5,
    "lines.markersize":   5,
    "patch.linewidth":    0.8,
    # Axes
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "axes.grid":          True,
    "grid.alpha":         0.35,
    "grid.linewidth":     0.5,
    # Output
    "figure.dpi":         150,          # screen preview
    "savefig.dpi":        300,          # saved files
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "pdf.fonttype":       42,           # embed fonts (required by most journals)
    "ps.fonttype":        42,
    "svg.fonttype":       "none",       # editable text in SVG/Illustrator
}
plt.rcParams.update(PUB_RC)


# ── Save helper ───────────────────────────────────────────────────────────────
# Default output folder — change FIGURE_DIR to any path you prefer.
FIGURE_DIR = "figures"

def _save_fig(fig, save_path, formats=("pdf", "png"), dpi=300):
    """
    Save `fig` in one or more publication formats.

    Parameters
    ----------
    fig       : matplotlib Figure
    save_path : str   Base path WITHOUT extension, e.g. 'figures/coassoc_matrix'.
                      One file is written per format listed in `formats`.
    formats   : tuple Subset of {'pdf', 'svg', 'png', 'eps', 'tiff'}.
                      pdf  — vector, font-embedded, preferred by most journals.
                      svg  — vector, editable in Illustrator / Inkscape.
                      png  — raster at `dpi` DPI (≥300 for publication).
                      eps  — legacy vector (PostScript), some journals require it.
                      tiff — high-res raster, common in biomedical journals.
    dpi       : int   Resolution for raster formats (png, tiff).  300 = standard,
                      600 = high-res microscopy / Nature-style figures.
    """
    # Create the parent directory if the caller passed a nested path.
    os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else ".", exist_ok=True)
    saved, skipped = [], []
    for fmt in formats:
        out = f"{save_path}.{fmt}"
        try:
            fig.savefig(out, format=fmt, dpi=dpi,
                        bbox_inches="tight", pad_inches=0.05)
            saved.append(out)
        except PermissionError:
            # File is open in another application (e.g. PDF viewer on Windows)
            # Warn and continue rather than losing the whole figure run.
            skipped.append(out)
            print(f"  WARNING: Could not write {out!r} — close the file and re-run if needed.")
    if saved:
        print(f"  Saved: {', '.join(saved)}")


print("Visualisation imports & publication style ready.")
print(f"Figures will be saved to: '{FIGURE_DIR}/' (change FIGURE_DIR to override)")

### 10b · Co-association heatmap

In [ ]:
def plot_coassoc(C, consensus_labels=None, figsize=(7, 6),
                 save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Publication-quality heatmap of the weighted co-association matrix.

    Parameters
    ----------
    C                : np.ndarray  N×N matrix from build_coassoc()
    consensus_labels : np.ndarray  optional — sorts samples and draws cluster
                                   boundary lines on the diagonal
    figsize          : tuple       inches; (7, 6) fits a single journal column
    save_path        : str or None base path without extension, e.g.
                                   'figures/fig1_coassoc'
                                   Pass None to skip saving.
    formats          : tuple       file formats to write, e.g. ('pdf', 'svg', 'png')
    dpi              : int         resolution for raster formats (300 minimum)
    """
    # This is the single most diagnostic figure of the ensemble: clean bright
    # blocks along the diagonal mean the algorithms agreed strongly about who
    # belongs together; a diffuse matrix means the consensus is weak.
    fig, ax = plt.subplots(figsize=figsize)

    if consensus_labels is not None:
        # Sorting by cluster is essential — in the original sample order the
        # block structure is invisible.
        order    = np.argsort(consensus_labels)
        C_plot   = C[np.ix_(order, order)]
        title_sfx = " — sorted by consensus cluster"
    else:
        C_plot   = C
        title_sfx = ""

    # vmin/vmax pinned to [0, 1] so brightness is comparable across runs.
    # rasterized=True keeps the PDF small despite N² pixels.
    im = ax.imshow(C_plot, cmap=CMAP_HEATMAP, vmin=0, vmax=1,
                   aspect="auto", interpolation="nearest",
                   rasterized=True)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Co-association weight", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    if consensus_labels is not None:
        sorted_labels       = consensus_labels[order]
        unique, counts      = np.unique(sorted_labels, return_counts=True)
        # Cumulative sizes give the pixel positions where one cluster ends.
        boundaries          = np.cumsum(counts)[:-1]
        for b in boundaries:
            ax.axhline(b - 0.5, color="crimson", lw=1.0, ls="--", alpha=0.8)
            ax.axvline(b - 0.5, color="crimson", lw=1.0, ls="--", alpha=0.8)
        # NOTE: these legend swatches come from tab10, not CLUSTER_PALETTE, so
        # they do not match the cluster colours used in the scatter figures.
        handles = [
            mpatches.Patch(
                facecolor=plt.colormaps[CMAP_CLUSTERS].resampled(max(len(unique), 2))(u),
                label=f"Cluster {u}  (n={c})"
            )
            for u, c in zip(unique, counts)
        ]
        ax.legend(handles=handles, loc="upper right",
                  fontsize=8, framealpha=0.9, edgecolor="0.7")

    ax.set_title(f"Co-association Matrix  (N={C.shape[0]}){title_sfx}", pad=10)
    ax.set_xlabel("Sample index")
    ax.set_ylabel("Sample index")
    # Individual sample indices are meaningless here, so hide the ticks.
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    # Remove top/right spines (already set globally, but explicit for heatmap)
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 10c · ARI agreement matrix

In [ ]:
def plot_ari_matrix(labels_dict, title="Pairwise ARI Agreement Matrix",
                    figsize=(6, 5), save_path=None,
                    formats=("pdf", "png"), dpi=300):
    """
    Annotated heatmap of pairwise Adjusted Rand Index between all algorithms.

    Parameters
    ----------
    labels_dict : dict  { algorithm_name: label_array }
    title       : str
    figsize     : tuple inches
    save_path   : str or None  base path without extension
    formats     : tuple        file formats to write
    dpi         : int          raster resolution
    """
    # Shows WHICH algorithms agree, not just how much on average — useful for
    # spotting an outlier method that is dragging the consensus.
    names = list(labels_dict.keys())
    M     = len(names)

    # Full M×M matrix (symmetric, diagonal = 1 by definition).
    A = np.array([
        [adjusted_rand_score(labels_dict[names[i]], labels_dict[names[j]])
         for j in range(M)]
        for i in range(M)
    ])

    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        A,
        xticklabels=names,
        yticklabels=names,
        annot=True,
        fmt=".2f",
        cmap=CMAP_ARI,
        # Slight negative floor: ARI can go below 0 when agreement is worse
        # than chance, and clipping at 0 would hide that.
        vmin=-0.1,
        vmax=1.0,
        linewidths=0.4,
        linecolor="white",
        annot_kws={"size": 8, "weight": "normal"},
        ax=ax,
        cbar_kws={
            "label": "Adjusted Rand Index",
            "fraction": 0.046,
            "pad": 0.04,
            "shrink": 0.85,
        }
    )

    ax.set_title(title, pad=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    # Thicken the diagonal border to highlight self-agreement
    # (the diagonal is trivially 1.0 and should be read as a reference, not data)
    for i in range(M):
        ax.add_patch(
            plt.Rectangle((i, i), 1, 1, fill=False,
                           edgecolor="black", lw=1.5)
        )

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

    # Summary excluding the trivial diagonal — the number worth quoting.
    mask = ~np.eye(M, dtype=bool)
    print(f"  Mean off-diagonal ARI : {A[mask].mean():.4f}")
    print(f"  Min  off-diagonal ARI : {A[mask].min():.4f}")

### 10d · t-SNE embedding (single label vector)

In [ ]:
# Embedding cache — shared by t-SNE and UMAP to avoid recomputation
# t-SNE on N≈4000 takes tens of seconds; every panel of every grid figure
# reuses the same embedding, so caching turns minutes into seconds.
_embed_cache = {}

def _array_key(X):
    """Stable hash for a numpy array (avoids id() reuse across GC cycles)."""
    # Content-based key: shape + dtype + a coarse checksum. Using id() would
    # risk a stale cache hit if numpy reused a freed memory address.
    return (X.shape, X.dtype, int(X.sum() * 1e6) % (2**31))


def _get_tsne(X, perplexity=30, random_state=42):
    """Compute (and cache) a 2-D t-SNE embedding of X.

    Opt-fix: cache key now uses a content hash instead of id() to prevent
    stale hits when numpy reuses memory addresses after garbage collection.
    """
    key = ("tsne", _array_key(X), perplexity, random_state)
    if key not in _embed_cache:
        print("  Computing t-SNE embedding ...", end=" ", flush=True)
        # sklearn renamed n_iter to max_iter in 1.4; detect at runtime so the
        # notebook works on either version without editing.
        import sklearn
        sk_version = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
        tsne_kwargs = dict(n_components=2, perplexity=perplexity,
                           random_state=random_state)
        tsne_kwargs["max_iter" if sk_version >= (1, 4) else "n_iter"] = 1000
        _embed_cache[key] = TSNE(**tsne_kwargs).fit_transform(X)
        print("done.")
    return _embed_cache[key]


def _get_umap(X, n_neighbors=15, min_dist=0.1, random_state=42):
    """Compute (and cache) a 2-D UMAP embedding of X.

    UMAP is typically 5-10x faster than t-SNE for large N and preserves
    global structure better.  Requires: pip install umap-learn
    """
    # Degrade gracefully rather than raising, so the notebook still completes.
    if not UMAP_AVAILABLE:
        print("  umap-learn not installed — falling back to t-SNE.")
        return _get_tsne(X, random_state=random_state)
    key = ("umap", _array_key(X), n_neighbors, min_dist, random_state)
    if key not in _embed_cache:
        print("  Computing UMAP embedding ...", end=" ", flush=True)
        reducer = umap_lib.UMAP(n_components=2, n_neighbors=n_neighbors,
                                min_dist=min_dist, random_state=random_state)
        _embed_cache[key] = reducer.fit_transform(X)
        print("done.")
    return _embed_cache[key]


def plot_embedding(X, labels, title="t-SNE Projection",
                   perplexity=30, random_state=42,
                   figsize=(5.5, 4.5),
                   save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Publication-quality 2-D t-SNE scatter plot coloured by cluster labels.

    Parameters
    ----------
    X            : np.ndarray  scaled feature matrix
    labels       : np.ndarray  integer cluster labels (-1 = noise)
    title        : str
    perplexity   : float
    random_state : int
    figsize      : tuple  inches; (5.5, 4.5) = one journal column
    save_path    : str or None  base path without extension
    formats      : tuple
    dpi          : int
    """
    # CAVEAT worth remembering when reading this figure: t-SNE preserves local
    # neighbourhoods, not global distances. Visual separation supports the
    # clustering, but the gaps between blobs are not meaningful magnitudes.
    emb        = _get_tsne(X, perplexity=perplexity, random_state=random_state)
    unique     = np.unique(labels)
    n_clusters = len(unique[unique != -1])

    fig, ax = plt.subplots(figsize=figsize)

    # One scatter call per cluster so each gets its own legend entry.
    for lab in unique:
        mask  = labels == lab
        color = "lightgray" if lab == -1 else _cluster_color(lab, n_clusters)
        lbl   = "Noise" if lab == -1 else f"Cluster {lab}  (n={mask.sum()})"
        ax.scatter(
            emb[mask, 0], emb[mask, 1],
            c=[color], s=18, alpha=0.72,     # alpha reveals density in dense regions
            edgecolors="none", label=lbl,
            rasterized=True          # rasterise points inside vector PDF
        )

    ax.legend(markerscale=1.8, framealpha=0.9, edgecolor="0.7",
              loc="best", ncol=max(1, n_clusters // 5))
    ax.set_title(f"{title}  —  K = {n_clusters}", pad=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    # t-SNE axis units are arbitrary, so numeric ticks would mislead.
    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 10e · t-SNE grid — all algorithms + consensus

In [ ]:
def plot_all_embeddings(X, labels_dict, consensus_labels,
                        perplexity=30, random_state=42,
                        n_cols=3, panel_size=(4.0, 3.2),
                        save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Publication-quality grid of t-SNE panels — one per algorithm + consensus.
    All panels share the same cached embedding.

    Parameters
    ----------
    X                : np.ndarray
    labels_dict      : dict        Stage-4 fixed-K label dict
    consensus_labels : np.ndarray
    perplexity       : float
    random_state     : int
    n_cols           : int         columns in the grid (default 3)
    panel_size       : tuple       (width, height) in inches per panel
    save_path        : str or None base path without extension
    formats          : tuple
    dpi              : int
    """
    # Critical design choice: every panel plots the SAME cached embedding, so
    # point positions are identical and only the colouring differs. Any visual
    # difference between panels is therefore a real labelling difference.
    emb       = _get_tsne(X, perplexity=perplexity, random_state=random_state)
    all_items = list(labels_dict.items()) + [("Consensus", consensus_labels)]
    n_panels  = len(all_items)
    n_rows    = int(np.ceil(n_panels / n_cols))   # enough rows to fit all panels

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(panel_size[0] * n_cols, panel_size[1] * n_rows),
        constrained_layout=True,
    )
    # Flatten so a single index works regardless of grid shape.
    axes = np.array(axes).flatten()

    for idx, (name, labels) in enumerate(all_items):
        ax     = axes[idx]
        unique = np.unique(labels)
        n_k    = len(unique[unique != -1])
        is_consensus = name == "Consensus"

        for lab in unique:
            mask  = labels == lab
            color = "lightgray" if lab == -1 else _cluster_color(lab, n_k)
            lbl   = "Noise" if lab == -1 else f"C{lab} (n={mask.sum()})"
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       c=[color], s=10, alpha=0.65,
                       edgecolors="none", label=lbl,
                       rasterized=True)

        ax.set_title(
            f"{name}  |  K={n_k}",
            fontsize=9,
            fontweight="bold" if is_consensus else "normal",
            color="navy" if is_consensus else "black",
            pad=5,
        )
        # Highlight consensus panel with a subtle border
        # so the reference partition is immediately identifiable in the grid.
        if is_consensus:
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor("navy")
                spine.set_linewidth(1.2)

        ax.legend(fontsize=6, markerscale=1.5, framealpha=0.85,
                  edgecolor="0.7", loc="best",
                  ncol=max(1, n_k // 4),
                  handletextpad=0.3, borderpad=0.4)
        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)
        ax.set_xlabel("t-SNE 1", fontsize=7)
        ax.set_ylabel("t-SNE 2", fontsize=7)

    # Hide unused grid slots when n_panels is not a multiple of n_cols.
    for j in range(n_panels, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("t-SNE Projections — All Algorithms & Consensus",
                 fontsize=11, fontweight="bold", y=1.01)

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 10e-b · UMAP embedding grid

In [ ]:
def plot_umap_grid(X, labels_dict, consensus_labels,
                   n_neighbors=15, min_dist=0.1, random_state=42,
                   n_cols=3, panel_size=(4.0, 3.2),
                   save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Publication-quality grid of UMAP panels — one per algorithm + consensus.
    All panels share the same cached UMAP embedding.

    Compared to t-SNE, UMAP better preserves global cluster distances and
    runs significantly faster on large datasets.

    Parameters
    ----------
    X                : np.ndarray  scaled feature matrix
    labels_dict      : dict        Stage-4 fixed-K labels
    consensus_labels : np.ndarray
    n_neighbors      : int         UMAP neighbourhood size (default 15)
    min_dist         : float       UMAP minimum distance (default 0.1)
    random_state     : int
    n_cols, panel_size, save_path, formats, dpi : see plot_all_embeddings
    """
    # Deliberately mirrors plot_all_embeddings so the two can be compared
    # panel-for-panel: structure appearing under BOTH embeddings is far more
    # convincing than structure visible under only one.
    emb       = _get_umap(X, n_neighbors=n_neighbors, min_dist=min_dist,
                           random_state=random_state)
    all_items = list(labels_dict.items()) + [("Consensus", consensus_labels)]
    n_panels  = len(all_items)
    n_rows    = int(np.ceil(n_panels / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(panel_size[0] * n_cols, panel_size[1] * n_rows),
        constrained_layout=True,
    )
    axes = np.array(axes).flatten()

    for idx, (name, labels) in enumerate(all_items):
        ax     = axes[idx]
        unique = np.unique(labels)
        n_k    = len(unique[unique != -1])
        is_consensus = name == "Consensus"

        for lab in unique:
            mask  = labels == lab
            color = "lightgray" if lab == -1 else _cluster_color(lab, n_k)
            lbl   = "Noise" if lab == -1 else f"C{lab} (n={mask.sum()})"
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       c=[color], s=10, alpha=0.65,
                       edgecolors="none", label=lbl,
                       rasterized=True)

        ax.set_title(
            f"{name}  |  K={n_k}",
            fontsize=9,
            fontweight="bold" if is_consensus else "normal",
            color="navy" if is_consensus else "black",
            pad=5,
        )
        if is_consensus:
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor("navy")
                spine.set_linewidth(1.2)

        ax.legend(fontsize=6, markerscale=1.5, framealpha=0.85,
                  edgecolor="0.7", loc="best",
                  ncol=max(1, n_k // 4),
                  handletextpad=0.3, borderpad=0.4)
        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)
        ax.set_xlabel("UMAP 1", fontsize=7)
        ax.set_ylabel("UMAP 2", fontsize=7)

    for j in range(n_panels, len(axes)):
        axes[j].set_visible(False)

    # Title states honestly which method actually produced the figure, since
    # _get_umap silently falls back to t-SNE when umap-learn is absent.
    method = "UMAP" if UMAP_AVAILABLE else "t-SNE (UMAP unavailable)"
    fig.suptitle(f"{method} Projections — All Algorithms & Consensus",
                 fontsize=11, fontweight="bold", y=1.01)

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()



def plot_umap_consensus(X, labels, title="UMAP Projection",
                         n_neighbors=15, min_dist=0.1, random_state=42,
                         figsize=(5.5, 4.5),
                         save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Publication-quality 2-D UMAP scatter plot coloured by cluster labels.
    Mirrors plot_embedding() (t-SNE) for direct side-by-side comparison.

    Uses the shared _embed_cache via _get_umap so this does not recompute
    the embedding if plot_umap_grid has already run.

    Parameters
    ----------
    X            : np.ndarray  scaled feature matrix
    labels       : np.ndarray  integer cluster labels (-1 = noise)
    title        : str
    n_neighbors  : int
    min_dist     : float
    random_state : int
    figsize      : tuple  inches; (5.5, 4.5) = one journal column
    save_path    : str or None  base path without extension
    formats      : tuple
    dpi          : int
    """
    # Standalone single-panel version, intended as the main-text companion to
    # the t-SNE consensus figure.
    emb        = _get_umap(X, n_neighbors=n_neighbors,
                            min_dist=min_dist,
                            random_state=random_state)
    unique     = np.unique(labels)
    n_clusters = len(unique[unique != -1])

    fig, ax = plt.subplots(figsize=figsize)

    for lab in unique:
        mask  = labels == lab
        color = "lightgray" if lab == -1 else _cluster_color(lab, n_clusters)
        lbl   = "Noise" if lab == -1 else f"Cluster {lab}  (n={mask.sum()})"
        ax.scatter(
            emb[mask, 0], emb[mask, 1],
            c=[color], s=18, alpha=0.72,
            edgecolors="none", label=lbl,
            rasterized=True,
        )

    ax.legend(markerscale=1.8, framealpha=0.9, edgecolor="0.7",
              loc="best", ncol=max(1, n_clusters // 5))
    ax.set_title(f"{title}  —  K = {n_clusters}", pad=8)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 10f · Cluster feature profiles

In [ ]:
def plot_cluster_profiles(df, labels, features,
                          title="Cluster Feature Profiles",
                          figsize=None,
                          save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Publication-quality parallel-coordinates plot of per-cluster feature
    medians with IQR shaded bands and raw-value annotations.

    Parameters
    ----------
    df       : pd.DataFrame  original (unscaled) dataframe
    labels   : np.ndarray    integer cluster labels
    features : list[str]     columns to plot (original scale)
    title    : str
    figsize  : tuple or None  auto-sized if None
    save_path: str or None    base path without extension
    formats  : tuple
    dpi      : int
    """
    # This is the figure that turns statistical clusters into clinical
    # phenotypes: the shape of each line is what justifies calling one cluster
    # SIDD and another SOIRD.
    temp_df = df[features].copy()
    temp_df["Cluster"] = labels
    unique  = sorted([c for c in np.unique(labels) if c != -1])
    n_k     = len(unique)

    # Widen automatically as more features are added, so labels never collide.
    if figsize is None:
        figsize = (max(6.0, len(features) * 1.3), 4.0)

    fig, ax = plt.subplots(figsize=figsize)
    x_pos   = np.arange(len(features))

    # Normalise each feature to [0, 1] across the whole dataset
    # so features with very different units share one vertical axis.
    # replace(0, 1) guards against divide-by-zero for a constant feature.
    feat_min   = temp_df[features].min()
    feat_max   = temp_df[features].max()
    feat_range = (feat_max - feat_min).replace(0, 1)

    for lab in unique:
        sub          = temp_df[temp_df["Cluster"] == lab][features]
        # Median line with an IQR band: robust central tendency plus a visual
        # sense of within-cluster spread and overlap between clusters.
        median_norm  = (sub.median()       - feat_min) / feat_range
        q25_norm     = (sub.quantile(0.25) - feat_min) / feat_range
        q75_norm     = (sub.quantile(0.75) - feat_min) / feat_range
        color        = _cluster_color(lab, n_k)

        ax.plot(x_pos, median_norm.values,
                marker="o", markersize=6, linewidth=1.8,
                color=color, label=f"Cluster {lab}  (n={len(sub)})",
                zorder=3, solid_capstyle="round")   # zorder keeps lines above bands

        ax.fill_between(x_pos, q25_norm.values, q75_norm.values,
                        alpha=0.12, color=color, zorder=2,
                        linewidth=0)

        # Raw-value annotation beside each point
        # Restores the clinical units that normalisation removed, so the reader
        # can see both the relative shape and the actual numbers.
        for fi, feat in enumerate(features):
            raw = sub[feat].median()
            ax.annotate(
                f"{raw:.1f}",
                xy=(fi, median_norm[feat]),
                xytext=(5, 3),
                textcoords="offset points",
                fontsize=7,
                color=color,
                alpha=0.9,
            )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(features)
    ax.set_ylabel("Normalised value  [0 – 1]")
    ax.set_ylim(-0.08, 1.12)   # headroom so annotations are not clipped
    ax.set_title(title, pad=10)
    ax.legend(framealpha=0.9, edgecolor="0.7", loc="upper right")
    ax.grid(axis="y", alpha=0.35, linewidth=0.5)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.spines["left"].set_linewidth(0.8)

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 10g · Run all visualisations

> **Run Cells 1–9 first** to produce `C`, `labels_dict`, `consensus_labels`,
> `new_label_dict`, `best_k`, `X_scaled`, `df`, and `features`.

In [ ]:
# ── Output folder (all figures written here) ─────────────────────────────────
# Change FIGURE_DIR at the top of Cell 35 to save elsewhere.
# (FIGURE_DIR is actually defined in the visualisation-imports cell above.)
os.makedirs(FIGURE_DIR, exist_ok=True)
print(f"Saving all figures to '{FIGURE_DIR}/' in PDF + PNG formats.\n")

# Formats to use for every figure:
#   ('pdf', 'png')        — standard: vector PDF + 300-DPI PNG
#   ('pdf', 'svg', 'png') — also SVG for Illustrator/Inkscape editing
#   ('tiff', 'eps')       — some biomedical journals require these
FMT = ("pdf", "png")
DPI = 300   # 300 standard | 600 high-res microscopy

# This cell is the single entry point that regenerates every main figure, so
# a reviewer revision only requires re-running one cell.

# ── 1. Co-association heatmap ─────────────────────────────────────────────────
print("Fig 1 — Co-association heatmap")
plot_coassoc(
    C,
    consensus_labels=consensus_labels,
    save_path=f"{FIGURE_DIR}/fig1_coassoc_matrix",
    formats=FMT, dpi=DPI,
)

# ── 2. ARI matrix — Stage-1 (auto-K) algorithms ──────────────────────────────
# Stage-1 agreement is expected to be LOWER because each algorithm used its
# own K; this panel documents the disagreement the ensemble had to resolve.
print("Fig 2a — ARI matrix (Stage-1 labels)")
plot_ari_matrix(
    labels_dict,
    title="ARI Agreement — Stage-1 Algorithms (auto-K)",
    save_path=f"{FIGURE_DIR}/fig2a_ari_stage1",
    formats=FMT, dpi=DPI,
)

# ── 3. ARI matrix — Stage-4 (fixed-K) algorithms ─────────────────────────────
# With K held constant, remaining disagreement reflects genuine algorithmic
# differences rather than differing cluster counts.
print("Fig 2b — ARI matrix (Stage-4 fixed-K labels)")
plot_ari_matrix(
    new_label_dict,
    title=f"ARI Agreement — Stage-4 Algorithms  (K={best_k})",
    save_path=f"{FIGURE_DIR}/fig2b_ari_stage4",
    formats=FMT, dpi=DPI,
)

# ── 4. t-SNE — consensus partition ───────────────────────────────────────────
print("Fig 3 — t-SNE consensus partition")
plot_embedding(
    X_scaled, consensus_labels,
    title=f"Consensus Partition",
    save_path=f"{FIGURE_DIR}/fig3_tsne_consensus",
    formats=FMT, dpi=DPI,
)

# ── 5. t-SNE grid — all algorithms ───────────────────────────────────────────
print("Fig 4 — t-SNE grid (all algorithms + consensus)")
plot_all_embeddings(
    X_scaled, new_label_dict, consensus_labels,
    save_path=f"{FIGURE_DIR}/fig4_tsne_grid",
    formats=FMT, dpi=DPI,
)

# ── 5b. UMAP grid — all algorithms ───────────────────────────────────────────
print("Fig 4b — UMAP grid (all algorithms + consensus)")
plot_umap_grid(
    X_scaled, new_label_dict, consensus_labels,
    save_path=f"{FIGURE_DIR}/fig4b_umap_grid",
    formats=FMT, dpi=DPI,
)

# ── 5c. UMAP — consensus partition (standalone) ──────────────────────────────
print("Fig 3b — UMAP consensus partition (standalone)")
plot_umap_consensus(
    X_scaled, consensus_labels,
    title="Consensus Partition",
    save_path=f"{FIGURE_DIR}/fig3b_umap_consensus",
    formats=FMT, dpi=DPI,
)

# ── 6. Cluster profiles — consensus partition ─────────────────────────────────
print("Fig 5 — Cluster profiles (consensus)")
plot_cluster_profiles(
    df, consensus_labels, features,
    title=f"Consensus Cluster Profiles  (K={best_k})",
    save_path=f"{FIGURE_DIR}/fig5_profiles_consensus",
    formats=FMT, dpi=DPI,
)

# ── 7. Cluster profiles — best-ranked algorithm ───────────────────────────────
# Side-by-side with Fig 5, this shows whether the top single algorithm
# reproduces the ensemble phenotypes or diverges from them.
best_algo = ranking_df.iloc[0]["Algorithm"]
print(f"Fig 6 — Cluster profiles ({best_algo})")
plot_cluster_profiles(
    df, new_label_dict[best_algo], features,
    title=f"Cluster Profiles — {best_algo}  (K={best_k})",
    save_path=f"{FIGURE_DIR}/fig6_profiles_{best_algo}",
    formats=FMT, dpi=DPI,
)

# ── Summary ───────────────────────────────────────────────────────────────────
# Inventory of what was written, as a quick check that nothing failed silently.
saved_files = sorted(
    [f for f in os.listdir(FIGURE_DIR)
     if f.endswith(tuple(FMT))]
)
print(f"\n{'─'*50}")
print(f"All done.  {len(saved_files)} files written to '{FIGURE_DIR}/':")
for fname in saved_files:
    size_kb = os.path.getsize(os.path.join(FIGURE_DIR, fname)) / 1024
    print(f"  {fname:<45}  {size_kb:6.1f} KB")

## 11 · Cluster Composition Analysis & Export

| Sub-section | What it produces |
|---|---|
| 11a | Build unified label DataFrame (all algorithms + consensus) |
| 11b | Raw counts & proportions table |
| 11c | `plot_stacked_props` — stacked-bar composition chart |
| 11d | `plot_cluster_size_heatmap` — cluster-size heatmap across algorithms |
| 11e | `plot_hungarian_agreement` — label-aligned agreement via Hungarian matching |
| 11f | Descriptive statistics per consensus cluster |
| 11g | Export consensus-labelled dataset to CSV |
| 11h | Run everything |

### 11a · Build unified label DataFrame

In [ ]:
# Add Consensus to the fixed-K label dict for joint analysis.
# We work on a copy so new_label_dict itself is never mutated.
# (Mutating it would corrupt the ranking and figure cells if they are re-run.)
combined_label_dict = dict(new_label_dict)           # Stage-4 algorithms
combined_label_dict["Consensus"] = consensus_labels.copy()

# rows = samples, columns = algorithm names
# This tidy layout is what the composition and agreement functions below expect.
df_labels = pd.DataFrame(combined_label_dict)

print(f"Label DataFrame shape : {df_labels.shape}  "
      f"(samples x algorithms)")
print(f"Algorithms            : {list(df_labels.columns)}")
df_labels.head()

### 11b · Raw counts & proportions

In [ ]:
def cluster_counts_and_props(df_labels):
    """
    Compute per-cluster sample counts and proportions for every algorithm.

    Returns
    -------
    counts_df : pd.DataFrame  shape (max_clusters, n_algorithms)
    props_df  : pd.DataFrame  same shape, column-normalised to [0, 1]
    """
    # Counts answer "how big is each cluster?"; proportions make algorithms
    # with different cluster counts directly comparable.
    counts = {
        alg: pd.Series(labels).value_counts().sort_index()
        for alg, labels in df_labels.items()
    }
    # fillna(0): an algorithm that never used a given label gets 0, not NaN,
    # so the table stays rectangular and integer-typed.
    counts_df = pd.DataFrame(counts).fillna(0).astype(int).sort_index()
    # Divide by the COLUMN total so each algorithm's proportions sum to 1.
    props_df  = counts_df.div(counts_df.sum(axis=0), axis=1)
    return counts_df, props_df


counts_df, props_df = cluster_counts_and_props(df_labels)

print("Raw counts per cluster / algorithm:")
print(counts_df.to_string())
print()
print("Proportions per cluster / algorithm:")
print(props_df.round(4).to_string())

### 11c · Stacked-bar composition chart

In [ ]:
def plot_stacked_props(props_df,
                       title="Cluster composition by algorithm",
                       figsize=(7, 4.5),
                       save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Stacked-bar chart showing what fraction of each algorithm's output
    belongs to each cluster label.

    Parameters
    ----------
    props_df  : pd.DataFrame  from cluster_counts_and_props() — shape
                              (n_clusters, n_algorithms), column-sums = 1
    title     : str
    figsize   : tuple  inches
    save_path : str or None  base path without extension
    formats   : tuple
    dpi       : int
    """
    # Reveals at a glance whether any algorithm produced a badly imbalanced
    # partition (one dominant band) versus the others.
    n_clusters = len(props_df.index)
    colors     = [
        plt.colormaps[CMAP_CLUSTERS].resampled(max(n_clusters, 2))(i)
        for i in range(n_clusters)
    ]

    fig, ax = plt.subplots(figsize=figsize)

    # `bottom` accumulates the running height so each cluster stacks on the
    # previous one rather than overwriting it.
    bottom = np.zeros(len(props_df.columns))
    for idx, (cluster_id, row) in enumerate(props_df.iterrows()):
        vals = row.values.astype(float)
        bars = ax.bar(
            props_df.columns,
            vals,
            bottom=bottom,
            color=colors[idx % len(colors)],   # modulo guards against K > palette size
            label=f"Cluster {cluster_id}",
            edgecolor="white",
            linewidth=0.6,
        )
        # Annotate each bar segment with its proportion (skip tiny segments)
        # — below ~4% the text would not fit inside the band.
        for bar, val, bot in zip(bars, vals, bottom):
            if val > 0.04:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bot + val / 2,             # vertical centre of this segment
                    f"{val:.2f}",
                    ha="center", va="center",
                    fontsize=7.5, color="white", fontweight="bold",
                )
        bottom += vals

    ax.set_xlabel("Algorithm")
    ax.set_ylabel("Proportion of samples")
    ax.set_ylim(0, 1.05)
    ax.set_title(title, pad=10)
    ax.set_xticklabels(props_df.columns, rotation=35, ha="right")
    # Display the axis as percentages while the underlying data stays in [0, 1].
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: f"{y:.0%}")
    )

    # Legend outside the axes so it never covers a bar.
    ax.legend(
        title="Cluster label",
        bbox_to_anchor=(1.02, 1), loc="upper left",
        framealpha=0.9, edgecolor="0.7",
    )

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 11d · Cluster-size heatmap across algorithms

In [ ]:
def plot_cluster_size_heatmap(counts_df,
                               title="Cluster sizes across algorithms",
                               figsize=(7, 3.5),
                               save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Heatmap of absolute cluster sizes: rows = cluster labels,
    columns = algorithms.  Helps spot algorithms that collapse clusters
    or produce highly imbalanced partitions.

    Parameters
    ----------
    counts_df : pd.DataFrame  from cluster_counts_and_props()
    title     : str
    figsize   : tuple
    save_path : str or None
    formats   : tuple
    dpi       : int
    """
    # Complements the stacked bars: absolute Ns matter because a cluster too
    # small to support a survival model is a practical problem downstream.
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        counts_df,
        annot=True,
        fmt="d",              # integer counts, no decimal places
        cmap="YlOrRd",        # sequential — counts have a natural zero
        linewidths=0.5,
        linecolor="white",
        ax=ax,
        cbar_kws={"label": "Sample count", "fraction": 0.046, "pad": 0.04},
        annot_kws={"size": 8},
    )

    ax.set_title(title, pad=10)
    ax.set_xlabel("Algorithm")
    ax.set_ylabel("Cluster label")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

### 11e · Label-aligned agreement matrix (Hungarian matching)

Raw ARI ignores label permutation, but this plot shows what percentage of
samples are assigned to the **same best-matched cluster** across every pair
of algorithms after optimal label re-mapping via the Hungarian algorithm.

In [ ]:
def _hungarian_align(labels_a, labels_b):
    """
    Find the optimal label permutation that maps labels_b onto labels_a
    using the Hungarian algorithm, then return the fraction of samples
    that agree after alignment.
    """
    # Why this exists alongside ARI: ARI gives a chance-corrected index that is
    # hard to communicate clinically. This returns a plain "X% of participants
    # were assigned to the same phenotype", which is far more interpretable.
    from scipy.optimize import linear_sum_assignment  # local guard
    classes_a = np.unique(labels_a)
    classes_b = np.unique(labels_b)
    # Square matrix sized to the larger label set so the assignment is valid
    # even when the two partitions have different cluster counts.
    n = max(len(classes_a), len(classes_b))

    # Contingency matrix (overlap counts)
    C = np.zeros((n, n), dtype=int)
    for i, ca in enumerate(classes_a):
        for j, cb in enumerate(classes_b):
            C[i, j] = np.sum((labels_a == ca) & (labels_b == cb))

    # Hungarian: maximise overlap → minimise negative overlap
    # (linear_sum_assignment only minimises, hence the sign flip.)
    row_ind, col_ind = linear_sum_assignment(-C)
    matched = C[row_ind, col_ind].sum()
    return matched / len(labels_a)


def plot_hungarian_agreement(df_labels,
                              title="Label-aligned agreement (Hungarian matching)",
                              figsize=(6, 5),
                              save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Pairwise heatmap of optimal-label-aligned agreement percentages.
    Higher = the two algorithms assign samples to the same cluster
    (after re-labelling) more often.

    Parameters
    ----------
    df_labels : pd.DataFrame  rows=samples, cols=algorithms (incl. Consensus)
    title     : str
    figsize   : tuple
    save_path : str or None
    formats   : tuple
    dpi       : int
    """
    names = list(df_labels.columns)
    M     = len(names)

    # Full matrix including the diagonal (trivially 100%) so the figure reads
    # as a conventional symmetric agreement matrix.
    H = np.zeros((M, M))
    for i in range(M):
        for j in range(M):
            H[i, j] = _hungarian_align(
                df_labels[names[i]].values,
                df_labels[names[j]].values,
            )

    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        H * 100,                    # display as percentage
        xticklabels=names,
        yticklabels=names,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        vmin=0, vmax=100,           # fixed scale: colours mean the same in every run
        linewidths=0.4,
        linecolor="white",
        ax=ax,
        cbar_kws={
            "label": "Agreement  (%)",
            "fraction": 0.046,
            "pad": 0.04,
        },
        annot_kws={"size": 8},
    )

    # Bold diagonal
    for i in range(M):
        ax.add_patch(
            plt.Rectangle((i, i), 1, 1, fill=False,
                           edgecolor="black", lw=1.5)
        )

    ax.set_title(title, pad=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()

    # Summary
    # Off-diagonal only — the diagonal would inflate every statistic to 100%.
    mask   = ~np.eye(M, dtype=bool)
    H_flat = H[mask] * 100
    print(f"  Mean pairwise agreement : {H_flat.mean():.1f}%")
    print(f"  Min  pairwise agreement : {H_flat.min():.1f}%")
    print(f"  Max  pairwise agreement : {H_flat.max():.1f}%")

### 11f · Descriptive statistics per consensus cluster

In [ ]:
def consensus_descriptive_stats(df, consensus_labels, features):
    """
    Per-cluster descriptive statistics (count, mean, std, median, IQR)
    for each feature, printed as a tidy table.

    Parameters
    ----------
    df               : pd.DataFrame  original (unscaled) dataframe
    consensus_labels : np.ndarray
    features         : list[str]
    """
    # This table is the basis of the manuscript's phenotype description table.
    # Computed on the ORIGINAL scale so values are clinically interpretable,
    # not on the log/standardised matrix used for clustering.
    temp = df[features].copy()
    temp["Cluster"] = consensus_labels

    rows = []
    unique = sorted(temp["Cluster"].unique())

    # Long format (one row per cluster x feature) rather than wide, so the
    # result is easy to filter, pivot or export.
    for clust in unique:
        sub = temp[temp["Cluster"] == clust][features]
        for feat in features:
            s = sub[feat]
            rows.append({
                "Cluster":  clust,
                "Feature":  feat,
                "N":        len(s),
                # Both mean/SD and median/IQR are reported: the skewed HOMA
                # variables are better summarised by the latter, but reviewers
                # frequently expect the former.
                "Mean":     round(s.mean(),  3),
                "SD":       round(s.std(),   3),
                "Median":   round(s.median(),3),
                "Q25":      round(s.quantile(0.25), 3),
                "Q75":      round(s.quantile(0.75), 3),
                "IQR":      round(s.quantile(0.75) - s.quantile(0.25), 3),
                "Min":      round(s.min(),   3),
                "Max":      round(s.max(),   3),
            })

    stats_df = pd.DataFrame(rows).set_index(["Cluster", "Feature"])
    return stats_df


stats_df = consensus_descriptive_stats(df, consensus_labels, features)

print("Descriptive statistics per consensus cluster:")
print(stats_df.to_string())

### 12 · Bootstrap confidence intervals on cluster medians

In [ ]:
## ── Bootstrap Confidence Intervals on Cluster Feature Statistics ─────────────
## For each consensus cluster × feature, we compute the 95% CI of the median
## via B=500 bootstrap resamples.  This quantifies how reliable the
## per-cluster medians are.

def bootstrap_cluster_medians(df, labels, features, B=500, ci=0.95,
                               random_state=42):
    """
    Bootstrap 95% CI of the per-cluster feature median.

    Returns a DataFrame with columns:
        Cluster, Feature, Median, CI_Low, CI_High, CI_Width
    """
    # Why bootstrap: the median has no simple closed-form standard error, and
    # the features are skewed, so a normal-theory CI would be wrong. Resampling
    # makes no distributional assumption.
    # NOTE: this treats cluster membership as FIXED and resamples only within
    # clusters, so it captures sampling variability in the median but NOT the
    # additional uncertainty in the clustering itself.
    rng  = np.random.RandomState(random_state)
    alpha = (1 - ci) / 2       # two-tailed: 0.025 each side for a 95% CI
    rows = []

    temp = df[features].copy()
    temp["Cluster"] = labels
    unique_clusters = sorted(temp["Cluster"].unique())

    for clust in unique_clusters:
        sub = temp[temp["Cluster"] == clust][features].values
        n   = len(sub)
        for fi, feat in enumerate(features):
            col = sub[:, fi]
            obs_med = np.median(col)
            # Resample WITH replacement at the original size n — the defining
            # property of the bootstrap.
            boot_meds = np.array([
                np.median(col[rng.randint(0, n, n)])
                for _ in range(B)
            ])
            # Percentile method: take empirical quantiles of the bootstrap
            # distribution as the interval bounds.
            ci_low  = float(np.percentile(boot_meds, 100 * alpha))
            ci_high = float(np.percentile(boot_meds, 100 * (1 - alpha)))
            rows.append({
                "Cluster":   clust,
                "Feature":   feat,
                "N":         n,
                "Median":    round(obs_med,  3),
                "CI_Low":    round(ci_low,   3),
                "CI_High":   round(ci_high,  3),
                # Narrow width = a precisely estimated cluster median; wide
                # width flags a small or highly variable cluster.
                "CI_Width":  round(ci_high - ci_low, 3),
            })

    return pd.DataFrame(rows).set_index(["Cluster", "Feature"])


print("Computing bootstrap 95% CIs on consensus cluster medians (B=500) ...")
ci_df = bootstrap_cluster_medians(df, consensus_labels, features, B=500)
print("\n95% Bootstrap CI of feature medians per consensus cluster:")
print(ci_df.to_string())

### 11g · Export consensus-labelled dataset

In [ ]:
# ── Attach consensus labels and all algorithm labels to the original df ────────
# Copy first so the in-memory df is never altered by the export step.
# NOTE: df already carries the Sensitivity_k* columns added in Stage 4b.
df_export = df.copy()
df_export["Consensus_Cluster"] = consensus_labels

# Optionally attach every Stage-4 algorithm label for downstream analysis
# so a reviewer can reproduce sensitivity analyses under any single algorithm
# without re-running this notebook.
for alg, lab in new_label_dict.items():
    df_export[f"Label_{alg}"] = lab

# ── CSV export ────────────────────────────────────────────────────────────────
# K is embedded in the filename so alternative runs never overwrite each other.
csv_name = f"consensus_k{best_k}_clusters.csv"
df_export.to_csv(csv_name, index=False)

print(f"Exported : {csv_name}")
print(f"Shape    : {df_export.shape}")
print()
print("Consensus cluster counts:")
print(df_export["Consensus_Cluster"].value_counts().sort_index().to_string())
print()

# ── WTMEC2YR verification ────────────────────────────────────────────────
# Final gate before the survival notebook: without the survey weight column
# the weighted sensitivity analysis cannot run at all.
if 'WTMEC2YR' in df_export.columns:
    missing_wt = df_export['WTMEC2YR'].isna().sum()
    print(f'✅ WTMEC2YR present in {csv_name}')
    print(f'   Valid weights : {len(df_export) - missing_wt:,}  '
          f'Missing : {missing_wt}')
    print(f'   Weight range  : {df_export["WTMEC2YR"].min():.1f} – '
          f'{df_export["WTMEC2YR"].max():,.1f}')
    print(f'   Total columns : {df_export.shape[1]}')
else:
    print(f'⚠️  WTMEC2YR NOT in {csv_name}')
    print('   Upstream pipeline fix required — see data loading cell above.')

## 13 · Statistical Validation — Kruskal-Wallis & Post-Hoc

In [ ]:
## ── Statistical Validation — Kruskal-Wallis + Dunn Post-Hoc ─────────────────
## Tests whether each feature differs significantly across consensus clusters.
## • Kruskal-Wallis: non-parametric omnibus test (no normality assumption).
## • Post-hoc pairwise Mann-Whitney U with Bonferroni correction.
## • Effect size: epsilon² (rank-based, analogous to eta² for ANOVA).

def kruskal_wallis_table(df, labels, features, alpha=0.05):
    """
    Run Kruskal-Wallis for each feature; compute epsilon² effect size.

    epsilon² = (H - k + 1) / (n - k)  where H = KW statistic, k = groups.

    Returns a DataFrame with H, p-value, epsilon², and significance flag.
    """
    # IMPORTANT CAVEAT: the clusters were DERIVED from these same features, so
    # significance here is close to guaranteed and is not evidence that the
    # clusters are real. The effect size (epsilon²) is the informative column:
    # it ranks which features actually drive the separation.
    rows = []
    temp = df[features].copy()
    temp["Cluster"] = labels
    unique_clusters = sorted(temp["Cluster"].unique())
    k = len(unique_clusters)

    for feat in features:
        # One array per cluster, unpacked into kruskal(*groups).
        groups = [temp.loc[temp["Cluster"] == c, feat].values
                  for c in unique_clusters]
        H, p = kruskal(*groups)
        n_total = sum(len(g) for g in groups)
        # max(0, ...) because the formula can go slightly negative when H is
        # small, which is not meaningful for a variance-explained measure.
        eps2 = max(0.0, (H - k + 1) / (n_total - k))
        rows.append({
            "Feature":    feat,
            "H_stat":     round(H,    3),
            "p_value":    p,
            "epsilon2":   round(eps2, 4),
            "Significant": "***" if p < 0.001 else ("**" if p < 0.01
                            else ("*" if p < alpha else "ns")),
        })

    return pd.DataFrame(rows).set_index("Feature")


def dunn_posthoc(df, labels, features, alpha=0.05):
    """
    Pairwise Mann-Whitney U tests with Bonferroni correction for each feature.

    Returns a dict { feature: DataFrame of pairwise p-values (adjusted) }.
    """
    # Kruskal-Wallis says "the clusters differ somewhere"; this identifies
    # WHICH pairs differ. Bonferroni is the conservative choice — with only
    # k(k-1)/2 comparisons the loss of power is acceptable.
    temp = df[features].copy()
    temp["Cluster"] = labels
    unique_clusters = sorted(temp["Cluster"].unique())
    k = len(unique_clusters)
    n_comparisons = k * (k - 1) // 2   # Bonferroni denominator

    result = {}
    for feat in features:
        groups = {c: temp.loc[temp["Cluster"] == c, feat].values
                  for c in unique_clusters}
        mat = pd.DataFrame(np.nan, index=unique_clusters, columns=unique_clusters)
        # Upper triangle only, then mirrored — avoids testing each pair twice.
        for i, ca in enumerate(unique_clusters):
            for cb in unique_clusters[i+1:]:
                _, p_raw = mannwhitneyu(groups[ca], groups[cb],
                                        alternative="two-sided")
                # min(1.0, ...) keeps the adjusted value a valid probability.
                p_adj = min(1.0, p_raw * n_comparisons)  # Bonferroni
                mat.loc[ca, cb] = round(p_adj, 4)
                mat.loc[cb, ca] = round(p_adj, 4)
        np.fill_diagonal(mat.values, 1.0)   # a cluster never differs from itself
        result[feat] = mat

    return result


print("=" * 60)
print("STATISTICAL VALIDATION")
print("=" * 60)

kw_table = kruskal_wallis_table(df, consensus_labels, features)
print("\nKruskal-Wallis test (consensus clusters):")
print(kw_table.to_string())
print("\nEffect size guide:  epsilon² < 0.01 trivial | 0.01–0.06 small |")
print("                    0.06–0.14 medium | > 0.14 large")

print("\n" + "─" * 60)
print("Post-hoc pairwise Mann-Whitney U (Bonferroni-corrected p-values):")
posthoc = dunn_posthoc(df, consensus_labels, features)
for feat, mat in posthoc.items():
    print(f"\n  {feat}:")
    print(mat.to_string())

### 13b · Boxplots with significance annotations

In [ ]:
def plot_cluster_boxplots(df, labels, features, posthoc_dict,
                           kw_table, figsize=None,
                           save_path=None, formats=("pdf", "png"), dpi=300):
    """
    Per-feature boxplots coloured by consensus cluster with:
      • Kruskal-Wallis p / epsilon² in the subtitle
      • Bonferroni-corrected pairwise significance bars

    Parameters
    ----------
    df           : pd.DataFrame  original (unscaled) data
    labels       : np.ndarray    cluster labels
    features     : list[str]
    posthoc_dict : dict          from dunn_posthoc()
    kw_table     : pd.DataFrame  from kruskal_wallis_table()
    """
    # Combines distribution, effect size and pairwise significance in one
    # figure — the format most clinical journals expect for this comparison.
    temp = df[features].copy()
    temp["Cluster"] = labels
    unique = sorted(temp["Cluster"].unique())
    n_k    = len(unique)
    n_feat = len(features)

    if figsize is None:
        figsize = (max(5.0, n_feat * 2.8), 4.5)

    # sharey=False because the features are on completely different scales.
    fig, axes = plt.subplots(1, n_feat, figsize=figsize, sharey=False)
    if n_feat == 1:
        axes = [axes]      # normalise to a list when matplotlib returns a bare Axes
    # NOTE: indexing a resampled colormap by 1-based cluster id can collide for
    # the highest cluster — see "Known issues" in the notebook header.
    colors = [plt.colormaps[CMAP_CLUSTERS].resampled(max(n_k, 2))(c)
              for c in unique]

    for ax, feat in zip(axes, features):
        data_by_cluster = [temp.loc[temp["Cluster"] == c, feat].values
                           for c in unique]
        bp = ax.boxplot(
            data_by_cluster,
            patch_artist=True,       # required so boxes can be colour-filled
            medianprops=dict(color="black", linewidth=1.5),
            whiskerprops=dict(linewidth=0.8),
            capprops=dict(linewidth=0.8),
            flierprops=dict(marker=".", markersize=3, alpha=0.4,
                            markerfacecolor="gray"),
            widths=0.55,
        )
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.75)

        # Significance bars
        # Anchored to the 95th percentile rather than the max, so a single
        # extreme outlier cannot push the annotations off the top of the panel.
        mat   = posthoc_dict[feat]
        y_max = max(np.percentile(d, 95) for d in data_by_cluster)
        y_rng = y_max - min(np.percentile(d, 5) for d in data_by_cluster)
        bar_h = y_rng * 0.06          # bar spacing scaled to the data range
        y_cur = y_max + bar_h * 0.5

        pairs = [(i, j) for i in range(n_k) for j in range(i+1, n_k)]
        for (i, j) in pairs:
            ca, cb = unique[i], unique[j]
            p = mat.loc[ca, cb]
            sig = "***" if p < 0.001 else ("**" if p < 0.01
                  else ("*" if p < 0.05 else ""))
            # Only draw a bracket when the pair is significant, keeping the
            # panel readable.
            if sig:
                x1, x2 = i + 1, j + 1     # boxplot positions are 1-based
                # Four points trace the bracket: up, across, down.
                ax.plot([x1, x1, x2, x2],
                        [y_cur, y_cur + bar_h * 0.3,
                         y_cur + bar_h * 0.3, y_cur],
                        lw=0.8, c="black")
                ax.text((x1 + x2) / 2, y_cur + bar_h * 0.35, sig,
                        ha="center", va="bottom", fontsize=8)
                y_cur += bar_h * 1.1      # stack the next bracket above
        kw_row = kw_table.loc[feat]
        ax.set_title(
            f"{feat}\nH={kw_row['H_stat']:.1f}, "
            f"ε²={kw_row['epsilon2']:.3f} {kw_row['Significant']}",
            fontsize=8, pad=4
        )
        ax.set_xticks(range(1, n_k + 1))
        ax.set_xticklabels([f"C{c}" for c in unique], fontsize=8)
        ax.set_xlabel("Cluster")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("Feature value (original scale)")
    fig.suptitle(
        f"Feature distributions per consensus cluster  "
        f"(K={n_k}, Bonferroni-corrected * p<.05, ** p<.01, *** p<.001)",
        fontsize=10, fontweight="bold", y=1.02
    )
    plt.tight_layout()

    if save_path:
        _save_fig(fig, save_path, formats=formats, dpi=dpi)

    plt.show()


print("Fig 10 — Cluster boxplots with significance")
plot_cluster_boxplots(
    df, consensus_labels, features,
    posthoc_dict=posthoc,
    kw_table=kw_table,
    save_path=f"{FIGURE_DIR}/fig10_cluster_boxplots",
    formats=FMT, dpi=DPI,
)

### 11h · Run full composition analysis

> Requires Cells 1–10 to have been executed first.

In [ ]:
# Second figure-generation entry point, covering the composition and
# statistical-validation figures (the first one, in Section 10g, covers the
# ensemble diagnostics). Re-declares FMT/DPI so this cell can run standalone.
os.makedirs(FIGURE_DIR, exist_ok=True)
FMT = ("pdf", "png")
DPI = 300

# ── Fig 8 : Stacked proportion chart ─────────────────────────────────────────
print("Fig 7 — Stacked proportions")
plot_stacked_props(
    props_df,
    title=f"Cluster Composition by Algorithm  (consensus K={best_k})",
    save_path=f"{FIGURE_DIR}/fig7_stacked_props",
    formats=FMT, dpi=DPI,
)

# ── Fig 9 : Cluster-size heatmap ──────────────────────────────────────────────
print("Fig 8 — Cluster-size heatmap")
plot_cluster_size_heatmap(
    counts_df,
    title=f"Cluster Sizes Across Algorithms  (consensus K={best_k})",
    save_path=f"{FIGURE_DIR}/fig8_cluster_size_heatmap",
    formats=FMT, dpi=DPI,
)

# ── Fig 10 : Hungarian agreement matrix ───────────────────────────────────────
print("Fig 9 — Hungarian label-aligned agreement matrix")
plot_hungarian_agreement(
    df_labels,
    title="Label-Aligned Agreement Matrix (Hungarian Matching)",
    save_path=f"{FIGURE_DIR}/fig9_hungarian_agreement",
    formats=FMT, dpi=DPI,
)

# ── Fig 10 : Cluster boxplots with statistical significance ───────────────────
# NOTE: this figure was already produced at the end of the boxplot definition
# cell; regenerating it here overwrites the identical file.
print("Fig 10 — Cluster boxplots with significance")
plot_cluster_boxplots(
    df, consensus_labels, features,
    posthoc_dict=posthoc,
    kw_table=kw_table,
    save_path=f"{FIGURE_DIR}/fig10_cluster_boxplots",
    formats=FMT, dpi=DPI,
)

# ── Summary of all figures ─────────────────────────────────────────────────────
# Lists everything in FIGURE_DIR, including figures from the earlier run cell.
saved_files = sorted([
    f for f in os.listdir(FIGURE_DIR)
    if f.endswith(tuple(FMT))
])
print(f"\n{'─'*55}")
print(f"All done.  {len(saved_files)} figure files in '{FIGURE_DIR}/':")
for fname in saved_files:
    kb = os.path.getsize(os.path.join(FIGURE_DIR, fname)) / 1024
    print(f"  {fname:<50}  {kb:6.1f} KB")